<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/18_Privacy_Fidelity_Utility_Trade_off.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ============================================================
# NOTEBOOK 18 — PRIVACY–FIDELITY–UTILITY TRADE-OFF
# SECTION 01 — HEADER & SCOPE
# ============================================================

import os
import gc
import json
import math
import random
import hashlib
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# 1. NOTEBOOK IDENTIFICATION
# ------------------------------------------------------------

NOTEBOOK_ID = "18"
NOTEBOOK_NAME = "Privacy–Fidelity–Utility Trade-off"

# ------------------------------------------------------------
# 2. REPRODUCIBILITY CONFIGURATION
# Aligned with the research methodology, Section 4.4.
# ------------------------------------------------------------

MASTER_SEED = 2025

REPETITIONS = [1, 2, 3, 4, 5]

REPETITION_SEEDS = {
    repetition: MASTER_SEED + 1000 + repetition
    for repetition in REPETITIONS
}

EXPECTED_REPETITION_SEEDS = {
    1: 3026,
    2: 3027,
    3: 3028,
    4: 3029,
    5: 3030
}

assert REPETITION_SEEDS == EXPECTED_REPETITION_SEEDS

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

# ------------------------------------------------------------
# 3. EXPERIMENTAL DATASETS
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

# ------------------------------------------------------------
# 4. ABLATION VARIANTS
# Aligned with the research methodology, Section 4.9.
# ------------------------------------------------------------

VARIANTS = [
    "gan_only",
    "statistical_guidance_gan",
    "gan_dp",
    "full_sppgan"
]

DP_VARIANTS = {
    "gan_dp",
    "full_sppgan"
}

# ------------------------------------------------------------
# 5. EXPERIMENTAL KEYS AND EXPECTED COVERAGE
# ------------------------------------------------------------

KEY_COLUMNS = [
    "dataset",
    "variant",
    "repetition_id"
]

EXPECTED_RECORDS = (
    len(DATASETS)
    * len(VARIANTS)
    * len(REPETITIONS)
)

EXPECTED_DP_RECORDS = (
    len(DATASETS)
    * len(REPETITIONS)
    * len(DP_VARIANTS)
)

EXPECTED_NON_DP_RECORDS = (
    len(DATASETS)
    * len(REPETITIONS)
    * (len(VARIANTS) - len(DP_VARIANTS))
)

# ------------------------------------------------------------
# 6. METHODOLOGY-ALIGNED PRIVACY CONFIGURATION
# Values represent configured targets, not achieved guarantees.
# ------------------------------------------------------------

DP_TARGET_EPSILON = 5.0
DP_DELTA_RULE = "min(1e-5, 1/N_train)"
DP_CLIPPING_NORM = 1.0
DP_ACCOUNTANT = "RDP"
DP_SAMPLING = "Poisson"
DP_PRIVATE_COMPONENT = "discriminator_critic"
DP_BATCH_SIZE = 128
DP_TRAINING_EPOCHS = 300

# ------------------------------------------------------------
# 7. EXPERIMENTAL DESIGN VALIDATION
# ------------------------------------------------------------

assert len(DATASETS) == 3
assert len(set(DATASETS)) == len(DATASETS)

assert len(VARIANTS) == 4
assert len(set(VARIANTS)) == len(VARIANTS)

assert len(REPETITIONS) == 5
assert len(set(REPETITIONS)) == len(REPETITIONS)

assert DP_VARIANTS.issubset(set(VARIANTS))

assert EXPECTED_RECORDS == 60
assert EXPECTED_DP_RECORDS == 30
assert EXPECTED_NON_DP_RECORDS == 30

# ------------------------------------------------------------
# 8. EXECUTION METADATA
# ------------------------------------------------------------

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

SECTION_01_VALIDATION = {
    "dataset_registry": len(DATASETS) == 3,
    "variant_registry": len(VARIANTS) == 4,
    "repetition_registry": len(REPETITIONS) == 5,
    "repetition_seeds": (
        REPETITION_SEEDS == EXPECTED_REPETITION_SEEDS
    ),
    "expected_experiment_grid": EXPECTED_RECORDS == 60,
    "dp_experiment_grid": EXPECTED_DP_RECORDS == 30,
    "non_dp_experiment_grid": EXPECTED_NON_DP_RECORDS == 30
}

SECTION_01_STATUS = (
    "PASS"
    if all(SECTION_01_VALIDATION.values())
    else "FAIL"
)

# ------------------------------------------------------------
# 9. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print(f"NOTEBOOK {NOTEBOOK_ID} — {NOTEBOOK_NAME}")
print("=" * 75)

print("Datasets                 :", len(DATASETS))
print("Variants                 :", len(VARIANTS))
print("Repetitions              :", len(REPETITIONS))
print("Expected records         :", EXPECTED_RECORDS)
print("Expected DP records      :", EXPECTED_DP_RECORDS)
print("Expected non-DP records  :", EXPECTED_NON_DP_RECORDS)

print("-" * 75)

print("Master seed              :", MASTER_SEED)
print("Repetition seeds         :", REPETITION_SEEDS)

print("-" * 75)

print("DP target epsilon        :", DP_TARGET_EPSILON)
print("DP delta rule            :", DP_DELTA_RULE)
print("DP clipping norm         :", DP_CLIPPING_NORM)
print("DP accountant            :", DP_ACCOUNTANT)
print("DP sampling               :", DP_SAMPLING)
print("DP private component     :", DP_PRIVATE_COMPONENT)
print("DP batch size            :", DP_BATCH_SIZE)
print("DP training epochs       :", DP_TRAINING_EPOCHS)

print("-" * 75)

print("Validation checks        :", len(SECTION_01_VALIDATION))
print(
    "Validation passed        :",
    sum(SECTION_01_VALIDATION.values())
)
print(
    "Validation failed        :",
    len(SECTION_01_VALIDATION)
    - sum(SECTION_01_VALIDATION.values())
)
print("Section 01 status        :", SECTION_01_STATUS)
print("Execution UTC            :", RUN_TIMESTAMP)

print("=" * 75)

NOTEBOOK 18 — Privacy–Fidelity–Utility Trade-off
Datasets                 : 3
Variants                 : 4
Repetitions              : 5
Expected records         : 60
Expected DP records      : 30
Expected non-DP records  : 30
---------------------------------------------------------------------------
Master seed              : 2025
Repetition seeds         : {1: 3026, 2: 3027, 3: 3028, 4: 3029, 5: 3030}
---------------------------------------------------------------------------
DP target epsilon        : 5.0
DP delta rule            : min(1e-5, 1/N_train)
DP clipping norm         : 1.0
DP accountant            : RDP
DP sampling               : Poisson
DP private component     : discriminator_critic
DP batch size            : 128
DP training epochs       : 300
---------------------------------------------------------------------------
Validation checks        : 7
Validation passed        : 7
Validation failed        : 0
Section 01 status        : PASS
Execution UTC            : 2026-11-

In [23]:
# ============================================================
# NOTEBOOK 18 — PRIVACY–FIDELITY–UTILITY TRADE-OFF
# SECTION 02 — LOAD FIDELITY RESULTS
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOKS_ROOT = PROJECT_ROOT / "results" / "notebooks"

NOTEBOOK_ROOT = NOTEBOOKS_ROOT / "notebook_18"

SOURCE_ROOT = NOTEBOOKS_ROOT

DIRS = {
    name: NOTEBOOK_ROOT / name
    for name in [
        "results",
        "tables",
        "figures",
        "metadata",
        "validation",
        "manifests",
        "logs"
    ]
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = DIRS["results"]
TABLES_DIR = DIRS["tables"]
FIGURES_DIR = DIRS["figures"]
METADATA_DIR = DIRS["metadata"]
VALIDATION_DIR = DIRS["validation"]
MANIFESTS_DIR = DIRS["manifests"]
LOGS_DIR = DIRS["logs"]

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

# ------------------------------------------------------------
# 2. EXPECTED EXPERIMENTAL GRID
# ------------------------------------------------------------

EXPERIMENT_GRID = pd.MultiIndex.from_product(
    [DATASETS, VARIANTS, REPETITIONS],
    names=KEY_COLUMNS
).to_frame(index=False)

assert len(EXPERIMENT_GRID) == EXPECTED_RECORDS
assert not EXPERIMENT_GRID.duplicated(KEY_COLUMNS).any()

# ------------------------------------------------------------
# 3. REUSABLE CSV LOADER
# ------------------------------------------------------------

def load_csv_candidates(paths, required_columns=None):
    """
    Load the first existing CSV with the required schema.

    Returns:
        DataFrame
        source path
        load status
    """

    required_columns = required_columns or []

    attempted_paths = []

    for path in paths:

        path = Path(path)
        attempted_paths.append(str(path))

        if not path.is_file():
            continue

        try:
            frame = pd.read_csv(
                path,
                low_memory=False
            )

            missing_columns = [
                column
                for column in required_columns
                if column not in frame.columns
            ]

            if missing_columns:
                continue

            return frame, str(path), "LOADED"

        except (
            OSError,
            pd.errors.ParserError,
            pd.errors.EmptyDataError,
            UnicodeDecodeError
        ):
            continue

    return (
        pd.DataFrame(columns=required_columns),
        None,
        "NOT_FOUND_OR_INVALID"
    )


# ------------------------------------------------------------
# 4. EXPERIMENTAL KEY NORMALIZATION
# ------------------------------------------------------------

def normalize_keys(frame):

    frame = frame.copy()

    if not set(KEY_COLUMNS).issubset(frame.columns):
        missing = set(KEY_COLUMNS) - set(frame.columns)

        raise ValueError(
            f"Missing experimental key columns: {missing}"
        )

    frame["dataset"] = (
        frame["dataset"]
        .astype("string")
        .str.strip()
    )

    frame["variant"] = (
        frame["variant"]
        .astype("string")
        .str.strip()
    )

    frame["repetition_id"] = pd.to_numeric(
        frame["repetition_id"],
        errors="coerce"
    ).astype("Int64")

    return frame


# ------------------------------------------------------------
# 5. FIDELITY SOURCE REGISTRY
# Notebook 17 is the primary source.
# ------------------------------------------------------------

FIDELITY_CANDIDATES = [
    SOURCE_ROOT
    / "notebook_17"
    / "results"
    / "fidelity_ablation_results.csv",

    SOURCE_ROOT
    / "notebook_17"
    / "results"
    / "ablation_fidelity_results.csv",

    SOURCE_ROOT
    / "notebook_17"
    / "results"
    / "fidelity_results.csv",

    SOURCE_ROOT
    / "notebook_16"
    / "results"
    / "fidelity_results.csv"
]

fidelity_raw, fidelity_source, fidelity_load_status = (
    load_csv_candidates(
        FIDELITY_CANDIDATES,
        required_columns=KEY_COLUMNS
    )
)

# ------------------------------------------------------------
# 6. FIDELITY METRIC REGISTRY
# Aligned with methodology Section 4.6.
# ------------------------------------------------------------

FIDELITY_METRIC_CANDIDATES = [
    "wasserstein_distance",
    "js_divergence",
    "correlation_error",
    "mmd",
    "ks_statistic",
    "ks_distance",
    "marginal_distance",
    "dependency_error",
    "categorical_divergence"
]

fidelity_metric_columns = [
    column
    for column in FIDELITY_METRIC_CANDIDATES
    if column in fidelity_raw.columns
]

# ------------------------------------------------------------
# 7. NORMALIZE AND VALIDATE LOADED RECORDS
# ------------------------------------------------------------

if fidelity_load_status == "LOADED":

    fidelity_raw = normalize_keys(fidelity_raw)

    fidelity_raw = fidelity_raw.loc[
        fidelity_raw["dataset"].isin(DATASETS)
        & fidelity_raw["variant"].isin(VARIANTS)
        & fidelity_raw["repetition_id"].isin(REPETITIONS)
    ].copy()

    invalid_key_records = int(
        fidelity_raw[KEY_COLUMNS].isna().any(axis=1).sum()
    )

    fidelity_raw = fidelity_raw.dropna(
        subset=KEY_COLUMNS
    ).copy()

else:
    invalid_key_records = 0

# ------------------------------------------------------------
# 8. DUPLICATE AND COVERAGE VALIDATION
# ------------------------------------------------------------

duplicate_records = int(
    fidelity_raw.duplicated(KEY_COLUMNS, keep=False).sum()
)

if duplicate_records:
    raise ValueError(
        "Duplicate fidelity experiment keys detected. "
        "Resolve duplicate records before trade-off analysis."
    )

observed_keys = (
    fidelity_raw[KEY_COLUMNS]
    .drop_duplicates()
    if not fidelity_raw.empty
    else pd.DataFrame(columns=KEY_COLUMNS)
)

coverage_check = EXPERIMENT_GRID.merge(
    observed_keys.assign(record_available=True),
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one"
)

coverage_check["record_available"] = (
    coverage_check["record_available"]
    .fillna(False)
    .astype(bool)
)

missing_fidelity_keys = coverage_check.loc[
    ~coverage_check["record_available"],
    KEY_COLUMNS
].copy()

available_fidelity_records = int(
    coverage_check["record_available"].sum()
)

missing_fidelity_records = (
    EXPECTED_RECORDS - available_fidelity_records
)

# ------------------------------------------------------------
# 9. METRIC VALIDATION
# ------------------------------------------------------------

if not fidelity_raw.empty and fidelity_metric_columns:

    for metric in fidelity_metric_columns:
        fidelity_raw[metric] = pd.to_numeric(
            fidelity_raw[metric],
            errors="coerce"
        )

    valid_metric_records = int(
        fidelity_raw[fidelity_metric_columns]
        .notna()
        .any(axis=1)
        .sum()
    )

else:
    valid_metric_records = 0

# ------------------------------------------------------------
# 10. SAVE LOADED SOURCE SNAPSHOT
# ------------------------------------------------------------

FIDELITY_SNAPSHOT_PATH = (
    RESULTS_DIR / "fidelity_source_snapshot.csv"
)

fidelity_raw.to_csv(
    FIDELITY_SNAPSHOT_PATH,
    index=False
)

# ------------------------------------------------------------
# 11. SAVE COVERAGE AND VALIDATION ARTIFACTS
# ------------------------------------------------------------

fidelity_coverage_df = pd.DataFrame([{
    "source_path": fidelity_source,
    "load_status": fidelity_load_status,
    "expected_records": EXPECTED_RECORDS,
    "available_records": available_fidelity_records,
    "missing_records": missing_fidelity_records,
    "invalid_key_records": invalid_key_records,
    "duplicate_records": duplicate_records,
    "available_metric_columns": len(fidelity_metric_columns),
    "valid_metric_records": valid_metric_records,
    "snapshot_path": str(FIDELITY_SNAPSHOT_PATH)
}])

fidelity_coverage_df.to_csv(
    VALIDATION_DIR / "fidelity_loading_validation.csv",
    index=False
)

missing_fidelity_keys.to_csv(
    VALIDATION_DIR / "missing_fidelity_experiments.csv",
    index=False
)

fidelity_source_metadata = {
    "source_path": fidelity_source,
    "load_status": fidelity_load_status,
    "metric_columns": fidelity_metric_columns,
    "expected_records": EXPECTED_RECORDS,
    "available_records": available_fidelity_records,
    "missing_records": missing_fidelity_records,
    "valid_metric_records": valid_metric_records,
    "snapshot_path": str(FIDELITY_SNAPSHOT_PATH)
}

with open(
    METADATA_DIR / "fidelity_source_metadata.json",
    "w"
) as file:
    json.dump(
        fidelity_source_metadata,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 12. SECTION STATUS
# ------------------------------------------------------------

FIDELITY_SCHEMA_VALID = (
    fidelity_load_status == "LOADED"
    and set(KEY_COLUMNS).issubset(fidelity_raw.columns)
)

FIDELITY_COVERAGE_COMPLETE = (
    available_fidelity_records == EXPECTED_RECORDS
)

FIDELITY_METRICS_AVAILABLE = (
    len(fidelity_metric_columns) > 0
    and valid_metric_records > 0
)

SECTION_02_VALIDATION = {
    "project_root": PROJECT_ROOT.is_dir(),
    "experiment_grid": len(EXPERIMENT_GRID) == EXPECTED_RECORDS,
    "source_schema": FIDELITY_SCHEMA_VALID,
    "duplicate_validation": duplicate_records == 0,
    "fidelity_metric_availability": FIDELITY_METRICS_AVAILABLE,
    "complete_experiment_coverage": FIDELITY_COVERAGE_COMPLETE,
    "snapshot_saved": FIDELITY_SNAPSHOT_PATH.is_file()
}

SECTION_02_STATUS = (
    "PASS"
    if all(SECTION_02_VALIDATION.values())
    else "INCOMPLETE"
)

# ------------------------------------------------------------
# 13. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 02 — FIDELITY RESULTS LOADING")
print("=" * 75)

print("Project root             :", PROJECT_ROOT)
print("Fidelity source          :", fidelity_source)
print("Source load status       :", fidelity_load_status)

print("-" * 75)

print("Expected experiments     :", EXPECTED_RECORDS)
print("Available records        :", available_fidelity_records)
print("Missing records          :", missing_fidelity_records)
print("Invalid key records      :", invalid_key_records)
print("Duplicate records        :", duplicate_records)

print("-" * 75)

print("Available fidelity metrics:", fidelity_metric_columns)
print("Valid metric records      :", valid_metric_records)

print("-" * 75)

print("Snapshot saved           :", FIDELITY_SNAPSHOT_PATH.is_file())
print("Coverage validation saved:", (
    VALIDATION_DIR / "fidelity_loading_validation.csv"
).is_file())

print("-" * 75)

print("Validation checks        :", len(SECTION_02_VALIDATION))
print(
    "Passed checks            :",
    sum(SECTION_02_VALIDATION.values())
)
print(
    "Failed checks            :",
    len(SECTION_02_VALIDATION)
    - sum(SECTION_02_VALIDATION.values())
)
print("Section 02 status        :", SECTION_02_STATUS)

print("=" * 75)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SECTION 02 — FIDELITY RESULTS LOADING
Project root             : /content/drive/MyDrive/SPP_GAN_Research
Fidelity source          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/results/fidelity_ablation_results.csv
Source load status       : LOADED
---------------------------------------------------------------------------
Expected experiments     : 60
Available records        : 0
Missing records          : 60
Invalid key records      : 0
Duplicate records        : 0
---------------------------------------------------------------------------
Available fidelity metrics: []
Valid metric records      : 0
---------------------------------------------------------------------------
Snapshot saved           : True
Coverage validation saved: True
---------------------------------------------------------------------------
Validation checks    

In [25]:
# ============================================================
# NOTEBOOK 18 — PRIVACY–FIDELITY–UTILITY TRADE-OFF
# SECTION 03 — LOAD UTILITY RESULTS
# ============================================================

# ------------------------------------------------------------
# 1. UTILITY SOURCE REGISTRY
# ------------------------------------------------------------

UTILITY_CANDIDATES = [
    SOURCE_ROOT / "notebook_17" / "results" / "utility_ablation_results.csv",
    SOURCE_ROOT / "notebook_17" / "results" / "ablation_utility_results.csv",
    SOURCE_ROOT / "notebook_17" / "results" / "utility_results.csv",
    SOURCE_ROOT / "notebook_16" / "results" / "utility_results.csv"
]

# ------------------------------------------------------------
# 2. LOAD UTILITY RESULTS
# Section 02 loader returns three values:
# DataFrame, source path, load status.
# ------------------------------------------------------------

utility_raw, utility_source, utility_load_status = (
    load_csv_candidates(
        UTILITY_CANDIDATES,
        required_columns=KEY_COLUMNS
    )
)

# ------------------------------------------------------------
# 3. TSTR METRIC REGISTRY
# Aligned with methodology Section 4.7.
# Primary classification metrics:
# Accuracy, Macro-F1, Weighted-F1.
# ------------------------------------------------------------

UTILITY_METRIC_CANDIDATES = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "tstr_accuracy",
    "tstr_macro_f1",
    "tstr_weighted_f1",
    "downstream_utility",
    "utility_score",
    "tstr_f1",
    "f1",
    "auc",
    "roc_auc",
    "r2",
    "rmse",
    "mae"
]

# ------------------------------------------------------------
# 4. NORMALIZE AND VALIDATE EXPERIMENTAL KEYS
# ------------------------------------------------------------

if utility_load_status == "LOADED":

    utility_raw = normalize_keys(utility_raw)

    utility_raw = utility_raw.loc[
        utility_raw["dataset"].isin(DATASETS)
        & utility_raw["variant"].isin(VARIANTS)
        & utility_raw["repetition_id"].isin(REPETITIONS)
    ].copy()

    invalid_utility_keys = int(
        utility_raw[KEY_COLUMNS].isna().any(axis=1).sum()
    )

    utility_raw = utility_raw.dropna(
        subset=KEY_COLUMNS
    ).copy()

else:
    utility_raw = pd.DataFrame(columns=KEY_COLUMNS)
    invalid_utility_keys = 0

# ------------------------------------------------------------
# 5. IDENTIFY AVAILABLE UTILITY METRICS
# ------------------------------------------------------------

available_utility_metrics = [
    column
    for column in UTILITY_METRIC_CANDIDATES
    if column in utility_raw.columns
]

for metric in available_utility_metrics:
    utility_raw[metric] = pd.to_numeric(
        utility_raw[metric],
        errors="coerce"
    )

valid_utility_records = (
    int(
        utility_raw[available_utility_metrics]
        .notna()
        .any(axis=1)
        .sum()
    )
    if available_utility_metrics
    else 0
)

# ------------------------------------------------------------
# 6. DUPLICATE VALIDATION
# ------------------------------------------------------------

duplicate_utility_records = int(
    utility_raw.duplicated(KEY_COLUMNS, keep=False).sum()
)

if duplicate_utility_records:
    raise ValueError(
        "Duplicate utility experiment keys detected. "
        "Resolve duplicates before trade-off analysis."
    )

# ------------------------------------------------------------
# 7. EXPERIMENTAL COVERAGE
# ------------------------------------------------------------

observed_utility_keys = (
    utility_raw[KEY_COLUMNS].drop_duplicates()
    if not utility_raw.empty
    else pd.DataFrame(columns=KEY_COLUMNS)
)

utility_coverage_check = EXPERIMENT_GRID.merge(
    observed_utility_keys.assign(record_available=True),
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one"
)

utility_coverage_check["record_available"] = (
    utility_coverage_check["record_available"]
    .fillna(False)
    .astype(bool)
)

available_utility_records = int(
    utility_coverage_check["record_available"].sum()
)

missing_utility_records = (
    EXPECTED_RECORDS - available_utility_records
)

missing_utility_keys = utility_coverage_check.loc[
    ~utility_coverage_check["record_available"],
    KEY_COLUMNS
].copy()

# ------------------------------------------------------------
# 8. SAVE UTILITY SOURCE SNAPSHOT
# ------------------------------------------------------------

UTILITY_SNAPSHOT_PATH = (
    RESULTS_DIR / "utility_source_snapshot.csv"
)

utility_raw.to_csv(
    UTILITY_SNAPSHOT_PATH,
    index=False
)

# ------------------------------------------------------------
# 9. SAVE COVERAGE AND VALIDATION ARTIFACTS
# ------------------------------------------------------------

utility_coverage_df = pd.DataFrame([{
    "source_path": utility_source,
    "load_status": utility_load_status,
    "expected_records": EXPECTED_RECORDS,
    "available_records": available_utility_records,
    "missing_records": missing_utility_records,
    "invalid_key_records": invalid_utility_keys,
    "duplicate_records": duplicate_utility_records,
    "available_metric_columns": len(available_utility_metrics),
    "valid_metric_records": valid_utility_records,
    "snapshot_path": str(UTILITY_SNAPSHOT_PATH)
}])

utility_coverage_df.to_csv(
    VALIDATION_DIR / "utility_loading_validation.csv",
    index=False
)

missing_utility_keys.to_csv(
    VALIDATION_DIR / "missing_utility_experiments.csv",
    index=False
)

utility_source_metadata = {
    "source_path": utility_source,
    "load_status": utility_load_status,
    "metric_columns": available_utility_metrics,
    "primary_methodology_metrics": [
        "accuracy",
        "macro_f1",
        "weighted_f1"
    ],
    "evaluation_protocol": "Train-on-Synthetic, Test-on-Real",
    "expected_records": EXPECTED_RECORDS,
    "available_records": available_utility_records,
    "missing_records": missing_utility_records,
    "valid_metric_records": valid_utility_records,
    "snapshot_path": str(UTILITY_SNAPSHOT_PATH)
}

with open(
    METADATA_DIR / "utility_source_metadata.json",
    "w"
) as file:
    json.dump(
        utility_source_metadata,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 10. SECTION VALIDATION
# ------------------------------------------------------------

UTILITY_SCHEMA_VALID = (
    utility_load_status == "LOADED"
    and set(KEY_COLUMNS).issubset(utility_raw.columns)
)

UTILITY_METRICS_AVAILABLE = (
    len(available_utility_metrics) > 0
    and valid_utility_records > 0
)

UTILITY_COVERAGE_COMPLETE = (
    available_utility_records == EXPECTED_RECORDS
)

SECTION_03_VALIDATION = {
    "source_schema": UTILITY_SCHEMA_VALID,
    "experiment_grid": len(EXPERIMENT_GRID) == EXPECTED_RECORDS,
    "duplicate_validation": duplicate_utility_records == 0,
    "utility_metric_availability": UTILITY_METRICS_AVAILABLE,
    "complete_experiment_coverage": UTILITY_COVERAGE_COMPLETE,
    "snapshot_saved": UTILITY_SNAPSHOT_PATH.is_file(),
    "coverage_report_saved": (
        VALIDATION_DIR / "utility_loading_validation.csv"
    ).is_file()
}

SECTION_03_STATUS = (
    "PASS"
    if all(SECTION_03_VALIDATION.values())
    else "INCOMPLETE"
)

# ------------------------------------------------------------
# 11. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 03 — UTILITY RESULTS LOADING")
print("=" * 75)

print("Utility source             :", utility_source)
print("Source load status         :", utility_load_status)

print("-" * 75)

print("Expected experiments       :", EXPECTED_RECORDS)
print("Available records          :", available_utility_records)
print("Missing records            :", missing_utility_records)
print("Invalid key records        :", invalid_utility_keys)
print("Duplicate records          :", duplicate_utility_records)

print("-" * 75)

print("Available utility metrics :", available_utility_metrics)
print("Valid metric records       :", valid_utility_records)

print("-" * 75)

print("Snapshot saved             :", UTILITY_SNAPSHOT_PATH.is_file())
print(
    "Coverage report saved      :",
    (VALIDATION_DIR / "utility_loading_validation.csv").is_file()
)

print("-" * 75)

print("Validation checks          :", len(SECTION_03_VALIDATION))
print(
    "Passed checks              :",
    sum(SECTION_03_VALIDATION.values())
)
print(
    "Failed checks              :",
    len(SECTION_03_VALIDATION)
    - sum(SECTION_03_VALIDATION.values())
)
print("Section 03 status          :", SECTION_03_STATUS)

print("=" * 75)

SECTION 03 — UTILITY RESULTS LOADING
Utility source             : None
Source load status         : NOT_FOUND_OR_INVALID
---------------------------------------------------------------------------
Expected experiments       : 60
Available records          : 0
Missing records            : 60
Invalid key records        : 0
Duplicate records          : 0
---------------------------------------------------------------------------
Available utility metrics : []
Valid metric records       : 0
---------------------------------------------------------------------------
Snapshot saved             : True
Coverage report saved      : True
---------------------------------------------------------------------------
Validation checks          : 7
Passed checks              : 4
Failed checks              : 3
Section 03 status          : INCOMPLETE


In [27]:
# ============================================================
# NOTEBOOK 18 — PRIVACY–FIDELITY–UTILITY TRADE-OFF
# SECTION 04 — LOAD PRIVACY RESULTS
# ============================================================

# ------------------------------------------------------------
# 1. PRIVACY SOURCE REGISTRY
# ------------------------------------------------------------

PRIVACY_CANDIDATES = [
    SOURCE_ROOT / "notebook_17" / "results" / "privacy_ablation_results.csv",
    SOURCE_ROOT / "notebook_17" / "results" / "ablation_privacy_results.csv",
    SOURCE_ROOT / "notebook_17" / "results" / "privacy_results.csv",
    SOURCE_ROOT / "notebook_16" / "results" / "privacy_results.csv"
]

# ------------------------------------------------------------
# 2. LOAD PRIVACY RESULTS
# Section 02 loader returns three values:
# DataFrame, source path, load status.
# ------------------------------------------------------------

privacy_raw, privacy_source, privacy_load_status = (
    load_csv_candidates(
        PRIVACY_CANDIDATES,
        required_columns=KEY_COLUMNS
    )
)

# ------------------------------------------------------------
# 3. EMPIRICAL DISCLOSURE-RISK METRIC REGISTRY
# Aligned with methodology Section 4.8.
# Formal DP accounting is loaded separately in Section 05.
# ------------------------------------------------------------

PRIVACY_RISK_CANDIDATES = [
    "nearest_neighbor_disclosure",
    "nearest_neighbor_risk",
    "record_linkage_risk",
    "reidentification_risk",
    "reidentification_rate",
    "membership_inference_auc",
    "mia_auc",
    "membership_inference_advantage",
    "mia_advantage",
    "disclosure_risk",
    "attribute_inference_accuracy"
]

# ------------------------------------------------------------
# 4. NORMALIZE EXPERIMENTAL KEYS
# ------------------------------------------------------------

if privacy_load_status == "LOADED":

    privacy_raw = normalize_keys(privacy_raw)

    privacy_raw = privacy_raw.loc[
        privacy_raw["dataset"].isin(DATASETS)
        & privacy_raw["variant"].isin(VARIANTS)
        & privacy_raw["repetition_id"].isin(REPETITIONS)
    ].copy()

    invalid_privacy_keys = int(
        privacy_raw[KEY_COLUMNS].isna().any(axis=1).sum()
    )

    privacy_raw = privacy_raw.dropna(
        subset=KEY_COLUMNS
    ).copy()

else:
    privacy_raw = pd.DataFrame(columns=KEY_COLUMNS)
    invalid_privacy_keys = 0

# ------------------------------------------------------------
# 5. IDENTIFY AND VALIDATE AVAILABLE METRICS
# ------------------------------------------------------------

available_privacy_metrics = [
    column
    for column in PRIVACY_RISK_CANDIDATES
    if column in privacy_raw.columns
]

for metric in available_privacy_metrics:
    privacy_raw[metric] = pd.to_numeric(
        privacy_raw[metric],
        errors="coerce"
    )

valid_privacy_records = (
    int(
        privacy_raw[available_privacy_metrics]
        .notna()
        .any(axis=1)
        .sum()
    )
    if available_privacy_metrics
    else 0
)

# ------------------------------------------------------------
# 6. DUPLICATE VALIDATION
# ------------------------------------------------------------

duplicate_privacy_records = int(
    privacy_raw.duplicated(KEY_COLUMNS, keep=False).sum()
)

if duplicate_privacy_records:
    raise ValueError(
        "Duplicate privacy experiment keys detected. "
        "Resolve duplicates before trade-off analysis."
    )

# ------------------------------------------------------------
# 7. EXPERIMENTAL COVERAGE
# ------------------------------------------------------------

observed_privacy_keys = (
    privacy_raw[KEY_COLUMNS].drop_duplicates()
    if not privacy_raw.empty
    else pd.DataFrame(columns=KEY_COLUMNS)
)

privacy_coverage_check = EXPERIMENT_GRID.merge(
    observed_privacy_keys.assign(record_available=True),
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one"
)

privacy_coverage_check["record_available"] = (
    privacy_coverage_check["record_available"]
    .fillna(False)
    .astype(bool)
)

available_privacy_records = int(
    privacy_coverage_check["record_available"].sum()
)

missing_privacy_records = (
    EXPECTED_RECORDS - available_privacy_records
)

missing_privacy_keys = privacy_coverage_check.loc[
    ~privacy_coverage_check["record_available"],
    KEY_COLUMNS
].copy()

# ------------------------------------------------------------
# 8. SAVE PRIVACY SOURCE SNAPSHOT
# ------------------------------------------------------------

PRIVACY_SNAPSHOT_PATH = (
    RESULTS_DIR / "privacy_source_snapshot.csv"
)

privacy_raw.to_csv(
    PRIVACY_SNAPSHOT_PATH,
    index=False
)

# ------------------------------------------------------------
# 9. SAVE COVERAGE AND VALIDATION ARTIFACTS
# ------------------------------------------------------------

privacy_coverage_df = pd.DataFrame([{
    "source_path": privacy_source,
    "load_status": privacy_load_status,
    "expected_records": EXPECTED_RECORDS,
    "available_records": available_privacy_records,
    "missing_records": missing_privacy_records,
    "invalid_key_records": invalid_privacy_keys,
    "duplicate_records": duplicate_privacy_records,
    "available_metric_columns": len(available_privacy_metrics),
    "valid_metric_records": valid_privacy_records,
    "snapshot_path": str(PRIVACY_SNAPSHOT_PATH)
}])

privacy_coverage_df.to_csv(
    VALIDATION_DIR / "privacy_loading_validation.csv",
    index=False
)

missing_privacy_keys.to_csv(
    VALIDATION_DIR / "missing_privacy_experiments.csv",
    index=False
)

privacy_source_metadata = {
    "source_path": privacy_source,
    "load_status": privacy_load_status,
    "metric_columns": available_privacy_metrics,
    "privacy_evaluation_type": "empirical_disclosure_risk",
    "formal_dp_accounting_source": "notebook_17",
    "expected_records": EXPECTED_RECORDS,
    "available_records": available_privacy_records,
    "missing_records": missing_privacy_records,
    "valid_metric_records": valid_privacy_records,
    "snapshot_path": str(PRIVACY_SNAPSHOT_PATH)
}

with open(
    METADATA_DIR / "privacy_source_metadata.json",
    "w"
) as file:
    json.dump(
        privacy_source_metadata,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 10. SECTION VALIDATION
# ------------------------------------------------------------

PRIVACY_SCHEMA_VALID = (
    privacy_load_status == "LOADED"
    and set(KEY_COLUMNS).issubset(privacy_raw.columns)
)

PRIVACY_METRICS_AVAILABLE = (
    len(available_privacy_metrics) > 0
    and valid_privacy_records > 0
)

PRIVACY_COVERAGE_COMPLETE = (
    available_privacy_records == EXPECTED_RECORDS
)

SECTION_04_VALIDATION = {
    "source_schema": PRIVACY_SCHEMA_VALID,
    "experiment_grid": len(EXPERIMENT_GRID) == EXPECTED_RECORDS,
    "duplicate_validation": duplicate_privacy_records == 0,
    "privacy_metric_availability": PRIVACY_METRICS_AVAILABLE,
    "complete_experiment_coverage": PRIVACY_COVERAGE_COMPLETE,
    "snapshot_saved": PRIVACY_SNAPSHOT_PATH.is_file(),
    "coverage_report_saved": (
        VALIDATION_DIR / "privacy_loading_validation.csv"
    ).is_file()
}

SECTION_04_STATUS = (
    "PASS"
    if all(SECTION_04_VALIDATION.values())
    else "INCOMPLETE"
)

# ------------------------------------------------------------
# 11. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 04 — PRIVACY RESULTS LOADING")
print("=" * 75)

print("Privacy source             :", privacy_source)
print("Source load status         :", privacy_load_status)

print("-" * 75)

print("Expected experiments       :", EXPECTED_RECORDS)
print("Available records          :", available_privacy_records)
print("Missing records            :", missing_privacy_records)
print("Invalid key records        :", invalid_privacy_keys)
print("Duplicate records          :", duplicate_privacy_records)

print("-" * 75)

print("Available privacy metrics :", available_privacy_metrics)
print("Valid metric records       :", valid_privacy_records)

print("-" * 75)

print("Snapshot saved             :", PRIVACY_SNAPSHOT_PATH.is_file())
print(
    "Coverage report saved      :",
    (VALIDATION_DIR / "privacy_loading_validation.csv").is_file()
)

print("-" * 75)

print("Validation checks          :", len(SECTION_04_VALIDATION))
print(
    "Passed checks              :",
    sum(SECTION_04_VALIDATION.values())
)
print(
    "Failed checks              :",
    len(SECTION_04_VALIDATION)
    - sum(SECTION_04_VALIDATION.values())
)
print("Section 04 status          :", SECTION_04_STATUS)

print("=" * 75)

SECTION 04 — PRIVACY RESULTS LOADING
Privacy source             : None
Source load status         : NOT_FOUND_OR_INVALID
---------------------------------------------------------------------------
Expected experiments       : 60
Available records          : 0
Missing records            : 60
Invalid key records        : 0
Duplicate records          : 0
---------------------------------------------------------------------------
Available privacy metrics : []
Valid metric records       : 0
---------------------------------------------------------------------------
Snapshot saved             : True
Coverage report saved      : True
---------------------------------------------------------------------------
Validation checks          : 7
Passed checks              : 4
Failed checks              : 3
Section 04 status          : INCOMPLETE


In [29]:
# ============================================================
# NOTEBOOK 18 — PRIVACY–FIDELITY–UTILITY TRADE-OFF
# SECTION 05 — LOAD DP ACCOUNTING
# ============================================================

# ------------------------------------------------------------
# 1. DP ACCOUNTING SOURCE REGISTRY
# ------------------------------------------------------------

DP_CANDIDATES = [
    SOURCE_ROOT / "notebook_17" / "results" / "dp_ablation_results.csv",
    SOURCE_ROOT / "notebook_17" / "results" / "ablation_dp_results.csv",
    SOURCE_ROOT / "notebook_17" / "results" / "dp_accounting_results.csv",
    SOURCE_ROOT / "notebook_17" / "metadata" / "dp_accounting_results.csv",
    SOURCE_ROOT / "notebook_16" / "results" / "dp_accounting_results.csv"
]

# ------------------------------------------------------------
# 2. METHODOLOGY-ALIGNED DP CONFIGURATION
# ------------------------------------------------------------

DP_TARGET_EPSILON = 5.0
DP_DELTA_RULE = "min(1e-5, 1/N_train)"
DP_CLIPPING_NORM = 1.0
DP_ACCOUNTANT = "RDP"
DP_SAMPLING = "Poisson"
DP_PRIVATE_COMPONENT = "discriminator_critic"
DP_BATCH_SIZE = 128
DP_TRAINING_EPOCHS = 300

# ------------------------------------------------------------
# 3. LOAD DP ACCOUNTING RESULTS
# Loader returns:
# DataFrame, source path, load status.
# ------------------------------------------------------------

dp_raw, dp_source, dp_load_status = load_csv_candidates(
    DP_CANDIDATES,
    required_columns=KEY_COLUMNS
)

# ------------------------------------------------------------
# 4. REQUIRED AND OPTIONAL ACCOUNTING FIELDS
# ------------------------------------------------------------

DP_ACCOUNTING_METRICS = [
    "achieved_epsilon",
    "epsilon",
    "delta",
    "noise_multiplier",
    "max_grad_norm",
    "sample_rate",
    "steps",
    "epochs",
    "batch_size",
    "dp_applied",
    "privacy_scope",
    "accountant",
    "private_component",
    "rdp_orders",
    "formal_dp_evidence"
]

# ------------------------------------------------------------
# 5. NORMALIZE EXPERIMENTAL KEYS
# ------------------------------------------------------------

if dp_load_status == "LOADED":

    dp_raw = normalize_keys(dp_raw)

    dp_raw = dp_raw.loc[
        dp_raw["dataset"].isin(DATASETS)
        & dp_raw["variant"].isin(VARIANTS)
        & dp_raw["repetition_id"].isin(REPETITIONS)
    ].copy()

    invalid_dp_keys = int(
        dp_raw[KEY_COLUMNS].isna().any(axis=1).sum()
    )

    dp_raw = dp_raw.dropna(
        subset=KEY_COLUMNS
    ).copy()

else:
    dp_raw = pd.DataFrame(columns=KEY_COLUMNS)
    invalid_dp_keys = 0

# ------------------------------------------------------------
# 6. NORMALIZE ACCOUNTING FIELD NAMES
# ------------------------------------------------------------

if "achieved_epsilon" not in dp_raw.columns and "epsilon" in dp_raw.columns:
    dp_raw["achieved_epsilon"] = dp_raw["epsilon"]

for column in DP_ACCOUNTING_METRICS:
    if column not in dp_raw.columns:
        dp_raw[column] = np.nan

NUMERIC_DP_COLUMNS = [
    "achieved_epsilon",
    "epsilon",
    "delta",
    "noise_multiplier",
    "max_grad_norm",
    "sample_rate",
    "steps",
    "epochs",
    "batch_size"
]

for column in NUMERIC_DP_COLUMNS:
    dp_raw[column] = pd.to_numeric(
        dp_raw[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 7. IDENTIFY CONFIGURED DP RUNS
# Configured DP is not equivalent to achieved DP.
# ------------------------------------------------------------

dp_raw["dp_configured"] = (
    dp_raw["variant"].isin(DP_VARIANTS)
)

if "dp_applied" in dp_raw.columns:
    dp_applied_text = (
        dp_raw["dp_applied"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    dp_raw["dp_applied_recorded"] = dp_applied_text.isin(
        ["true", "1", "yes"]
    )
else:
    dp_raw["dp_applied_recorded"] = False

# ------------------------------------------------------------
# 8. VALIDATE FORMAL ACCOUNTING EVIDENCE
# ------------------------------------------------------------

def valid_nonempty_text(series):
    return (
        series.notna()
        & series.astype(str).str.strip().ne("")
        & ~series.astype(str).str.lower().isin(
            ["nan", "none", "null", ""]
        )
    )

dp_raw["valid_epsilon"] = (
    dp_raw["achieved_epsilon"].gt(0)
    & np.isfinite(dp_raw["achieved_epsilon"])
)

dp_raw["valid_delta"] = (
    dp_raw["delta"].gt(0)
    & dp_raw["delta"].lt(1)
    & np.isfinite(dp_raw["delta"])
)

dp_raw["accountant_recorded"] = (
    valid_nonempty_text(dp_raw["accountant"])
    & dp_raw["accountant"].astype(str).str.upper().eq("RDP")
)

dp_raw["privacy_scope_recorded"] = valid_nonempty_text(
    dp_raw["privacy_scope"]
)

dp_raw["private_component_recorded"] = valid_nonempty_text(
    dp_raw["private_component"]
)

dp_raw["valid_accounting_evidence"] = (
    dp_raw["dp_configured"]
    & dp_raw["dp_applied_recorded"]
    & dp_raw["valid_epsilon"]
    & dp_raw["valid_delta"]
    & dp_raw["accountant_recorded"]
    & dp_raw["privacy_scope_recorded"]
    & dp_raw["private_component_recorded"]
)

dp_raw["within_target_epsilon"] = (
    dp_raw["valid_accounting_evidence"]
    & dp_raw["achieved_epsilon"].le(DP_TARGET_EPSILON)
)

# ------------------------------------------------------------
# 9. PRIVACY BOUNDARY INTERPRETATION
# Critic-only DP-SGD does not automatically establish
# end-to-end DP when statistical profiles, preprocessing,
# or conditional sampling are data-dependent.
# ------------------------------------------------------------

dp_raw["privacy_claim_scope"] = np.where(
    dp_raw["valid_accounting_evidence"],
    "ACCOUNTED_COMPONENT_ONLY_REVIEW_REQUIRED",
    "NOT_VERIFIED"
)

# ------------------------------------------------------------
# 10. DUPLICATE VALIDATION
# ------------------------------------------------------------

duplicate_dp_records = int(
    dp_raw.duplicated(KEY_COLUMNS, keep=False).sum()
)

if duplicate_dp_records:
    raise ValueError(
        "Duplicate DP accounting experiment keys detected. "
        "Resolve duplicates before privacy trade-off analysis."
    )

# ------------------------------------------------------------
# 11. EXPERIMENTAL COVERAGE
# ------------------------------------------------------------

observed_dp_keys = (
    dp_raw[KEY_COLUMNS].drop_duplicates()
    if not dp_raw.empty
    else pd.DataFrame(columns=KEY_COLUMNS)
)

dp_coverage_check = EXPERIMENT_GRID.merge(
    observed_dp_keys.assign(record_available=True),
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one"
)

dp_coverage_check["record_available"] = (
    dp_coverage_check["record_available"]
    .fillna(False)
    .astype(bool)
)

available_dp_records = int(
    dp_coverage_check["record_available"].sum()
)

missing_dp_records = (
    EXPECTED_RECORDS - available_dp_records
)

missing_dp_keys = dp_coverage_check.loc[
    ~dp_coverage_check["record_available"],
    KEY_COLUMNS
].copy()

# ------------------------------------------------------------
# 12. DP ACCOUNTING COVERAGE
# ------------------------------------------------------------

configured_dp_records = int(
    dp_raw["dp_configured"].sum()
)

formal_dp_evidence_records = int(
    dp_raw["valid_accounting_evidence"].sum()
)

within_target_records = int(
    dp_raw["within_target_epsilon"].sum()
)

expected_dp_runs = EXPECTED_DP_RECORDS

# ------------------------------------------------------------
# 13. SAVE DP ACCOUNTING SNAPSHOT
# ------------------------------------------------------------

DP_SNAPSHOT_PATH = (
    RESULTS_DIR / "dp_accounting_source_snapshot.csv"
)

dp_raw.to_csv(
    DP_SNAPSHOT_PATH,
    index=False
)

# ------------------------------------------------------------
# 14. SAVE VALIDATION ARTIFACTS
# ------------------------------------------------------------

dp_coverage_df = pd.DataFrame([{
    "source_path": dp_source,
    "load_status": dp_load_status,
    "expected_experiments": EXPECTED_RECORDS,
    "available_records": available_dp_records,
    "missing_records": missing_dp_records,
    "expected_dp_runs": expected_dp_runs,
    "configured_dp_records": configured_dp_records,
    "formal_accounting_evidence": formal_dp_evidence_records,
    "within_target_epsilon": within_target_records,
    "invalid_key_records": invalid_dp_keys,
    "duplicate_records": duplicate_dp_records,
    "snapshot_path": str(DP_SNAPSHOT_PATH)
}])

dp_coverage_df.to_csv(
    VALIDATION_DIR / "dp_accounting_loading_validation.csv",
    index=False
)

missing_dp_keys.to_csv(
    VALIDATION_DIR / "missing_dp_experiments.csv",
    index=False
)

dp_metadata = {
    "source_path": dp_source,
    "load_status": dp_load_status,
    "target_epsilon": DP_TARGET_EPSILON,
    "delta_rule": DP_DELTA_RULE,
    "clipping_norm": DP_CLIPPING_NORM,
    "accountant": DP_ACCOUNTANT,
    "sampling": DP_SAMPLING,
    "private_component": DP_PRIVATE_COMPONENT,
    "batch_size": DP_BATCH_SIZE,
    "training_epochs": DP_TRAINING_EPOCHS,
    "expected_dp_runs": expected_dp_runs,
    "configured_dp_records": configured_dp_records,
    "formal_accounting_evidence": formal_dp_evidence_records,
    "within_target_epsilon": within_target_records,
    "privacy_boundary_warning": (
        "Critic-only DP-SGD does not automatically establish "
        "end-to-end DP for data-dependent preprocessing, "
        "statistical guidance, or conditional sampling."
    ),
    "snapshot_path": str(DP_SNAPSHOT_PATH)
}

with open(
    METADATA_DIR / "dp_accounting_metadata.json",
    "w"
) as file:
    json.dump(dp_metadata, file, indent=2)

# ------------------------------------------------------------
# 15. SECTION VALIDATION
# ------------------------------------------------------------

DP_SCHEMA_VALID = (
    dp_load_status == "LOADED"
    and set(KEY_COLUMNS).issubset(dp_raw.columns)
)

DP_ACCOUNTING_AVAILABLE = (
    formal_dp_evidence_records > 0
)

DP_EVIDENCE_COMPLETE = (
    formal_dp_evidence_records == expected_dp_runs
)

SECTION_05_VALIDATION = {
    "experiment_grid": len(EXPERIMENT_GRID) == EXPECTED_RECORDS,
    "source_schema": DP_SCHEMA_VALID,
    "duplicate_validation": duplicate_dp_records == 0,
    "dp_accounting_evidence": DP_ACCOUNTING_AVAILABLE,
    "complete_dp_evidence": DP_EVIDENCE_COMPLETE,
    "snapshot_saved": DP_SNAPSHOT_PATH.is_file(),
    "coverage_report_saved": (
        VALIDATION_DIR / "dp_accounting_loading_validation.csv"
    ).is_file()
}

SECTION_05_STATUS = (
    "PASS"
    if all(SECTION_05_VALIDATION.values())
    else "INCOMPLETE"
)

# ------------------------------------------------------------
# 16. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 05 — DIFFERENTIAL PRIVACY ACCOUNTING")
print("=" * 75)

print("DP source                  :", dp_source)
print("Source load status         :", dp_load_status)

print("-" * 75)

print("Expected experiments      :", EXPECTED_RECORDS)
print("Available DP records      :", available_dp_records)
print("Missing records           :", missing_dp_records)
print("Expected DP runs          :", expected_dp_runs)
print("Configured DP records     :", configured_dp_records)

print("-" * 75)

print("Formal accounting evidence:", formal_dp_evidence_records)
print("Within target epsilon     :", within_target_records)
print("Target epsilon            :", DP_TARGET_EPSILON)
print("Delta rule                :", DP_DELTA_RULE)
print("Accountant                :", DP_ACCOUNTANT)
print("Sampling                  :", DP_SAMPLING)
print("Private component         :", DP_PRIVATE_COMPONENT)

print("-" * 75)

print("Snapshot saved            :", DP_SNAPSHOT_PATH.is_file())
print(
    "Coverage report saved     :",
    (VALIDATION_DIR / "dp_accounting_loading_validation.csv").is_file()
)

print("-" * 75)

print("Validation checks         :", len(SECTION_05_VALIDATION))
print(
    "Passed checks             :",
    sum(SECTION_05_VALIDATION.values())
)
print(
    "Failed checks             :",
    len(SECTION_05_VALIDATION)
    - sum(SECTION_05_VALIDATION.values())
)
print("Section 05 status         :", SECTION_05_STATUS)

print("=" * 75)

SECTION 05 — DIFFERENTIAL PRIVACY ACCOUNTING
DP source                  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/results/dp_ablation_results.csv
Source load status         : LOADED
---------------------------------------------------------------------------
Expected experiments      : 60
Available DP records      : 60
Missing records           : 0
Expected DP runs          : 30
Configured DP records     : 30
---------------------------------------------------------------------------
Formal accounting evidence: 0
Within target epsilon     : 0
Target epsilon            : 5.0
Delta rule                : min(1e-5, 1/N_train)
Accountant                : RDP
Sampling                  : Poisson
Private component         : discriminator_critic
---------------------------------------------------------------------------
Snapshot saved            : True
Coverage report saved     : True
---------------------------------------------------------------------------
Valida

In [31]:
# ============================================================
# NOTEBOOK 18 — PRIVACY–FIDELITY–UTILITY TRADE-OFF
# SECTION 06 — ALIGN EXPERIMENTAL CONDITIONS
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE SOURCE DATAFRAMES
# ------------------------------------------------------------

SOURCE_FRAMES = {
    "fidelity": fidelity_raw,
    "utility": utility_raw,
    "privacy": privacy_raw,
    "dp": dp_raw
}

# ------------------------------------------------------------
# 2. DEFINE METHODOLOGY-ALIGNED METRIC REGISTRIES
# ------------------------------------------------------------

SOURCE_METRICS = {
    "fidelity": [
        "wasserstein_distance",
        "js_divergence",
        "correlation_error",
        "mmd",
        "ks_statistic",
        "ks_distance",
        "marginal_distance",
        "dependency_error",
        "categorical_divergence",
        "ks_similarity",
        "categorical_similarity",
        "fidelity_score"
    ],

    "utility": [
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "tstr_accuracy",
        "tstr_macro_f1",
        "tstr_weighted_f1",
        "downstream_utility",
        "utility_score",
        "tstr_f1",
        "f1",
        "auc",
        "roc_auc",
        "r2",
        "rmse",
        "mae"
    ],

    "privacy": [
        "nearest_neighbor_disclosure",
        "nearest_neighbor_risk",
        "record_linkage_risk",
        "reidentification_risk",
        "reidentification_rate",
        "membership_inference_auc",
        "mia_auc",
        "membership_inference_advantage",
        "mia_advantage",
        "disclosure_risk",
        "attribute_inference_accuracy"
    ],

    "dp": [
        "dp_applied",
        "dp_configured",
        "achieved_epsilon",
        "epsilon",
        "delta",
        "noise_multiplier",
        "max_grad_norm",
        "sample_rate",
        "steps",
        "epochs",
        "batch_size",
        "privacy_scope",
        "accountant",
        "private_component",
        "rdp_orders",
        "formal_dp_evidence",
        "valid_accounting_evidence",
        "within_target_epsilon",
        "privacy_claim_scope"
    ]
}

# ------------------------------------------------------------
# 3. SOURCE PREPARATION FUNCTION
# Preserve actual source metrics and avoid fabricated values.
# ------------------------------------------------------------

def prepare_source(frame, source_name):

    if source_name not in SOURCE_METRICS:
        raise ValueError(
            f"Unknown source name: {source_name}"
        )

    if frame is None:
        frame = pd.DataFrame(columns=KEY_COLUMNS)

    frame = frame.copy()

    if frame.empty:
        return pd.DataFrame(
            columns=KEY_COLUMNS + SOURCE_METRICS[source_name]
        )

    frame = normalize_keys(frame)

    frame = frame.loc[
        frame["dataset"].isin(DATASETS)
        & frame["variant"].isin(VARIANTS)
        & frame["repetition_id"].isin(REPETITIONS)
    ].copy()

    if frame[KEY_COLUMNS].isna().any().any():
        raise ValueError(
            f"{source_name}: missing experimental keys detected."
        )

    if frame.duplicated(KEY_COLUMNS).any():
        raise ValueError(
            f"{source_name}: duplicate experimental keys detected. "
            "Resolve repeated measurements before alignment."
        )

    selected_metrics = [
        column
        for column in SOURCE_METRICS[source_name]
        if column in frame.columns
    ]

    result = frame[
        KEY_COLUMNS + selected_metrics
    ].copy()

    # Ensure stable schema without inventing measurements.
    for column in SOURCE_METRICS[source_name]:
        if column not in result.columns:
            result[column] = np.nan

    return result[
        KEY_COLUMNS + SOURCE_METRICS[source_name]
    ]


# ------------------------------------------------------------
# 4. PREPARE ALL SOURCES
# ------------------------------------------------------------

prepared = {
    name: prepare_source(frame, name)
    for name, frame in SOURCE_FRAMES.items()
}

# ------------------------------------------------------------
# 5. CHECK COLUMN COLLISIONS BEFORE MERGING
# ------------------------------------------------------------

seen_columns = set(KEY_COLUMNS)

for name, frame in prepared.items():

    metric_columns = [
        column
        for column in frame.columns
        if column not in KEY_COLUMNS
    ]

    collisions = seen_columns.intersection(metric_columns)

    if collisions:
        raise ValueError(
            f"Column collision in {name}: {sorted(collisions)}. "
            "Use explicit source-specific names before merging."
        )

    seen_columns.update(metric_columns)

# ------------------------------------------------------------
# 6. BUILD COMPLETE EXPERIMENT GRID
# ------------------------------------------------------------

aligned_df = EXPERIMENT_GRID.copy()

for name, frame in prepared.items():

    aligned_df = aligned_df.merge(
        frame,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# 7. ATTACH EXPERIMENTAL DESIGN INDICATORS
# ------------------------------------------------------------

aligned_df["dp_configured"] = (
    aligned_df["variant"].isin(DP_VARIANTS)
)

aligned_df["statistical_guidance_configured"] = (
    aligned_df["variant"].isin([
        "statistical_guidance_gan",
        "full_sppgan"
    ])
)

# Configured DP does not imply an achieved formal guarantee.
if "valid_accounting_evidence" in aligned_df.columns:
    aligned_df["formal_dp_verified"] = (
        aligned_df["valid_accounting_evidence"]
        .fillna(False)
        .astype(bool)
    )
else:
    aligned_df["formal_dp_verified"] = False

# ------------------------------------------------------------
# 8. VALIDATE EXPERIMENTAL ALIGNMENT
# ------------------------------------------------------------

assert len(aligned_df) == EXPECTED_RECORDS

assert not aligned_df.duplicated(KEY_COLUMNS).any()

assert aligned_df[KEY_COLUMNS].notna().all().all()

assert set(aligned_df["dataset"].unique()) == set(DATASETS)

assert set(aligned_df["variant"].unique()) == set(VARIANTS)

assert set(aligned_df["repetition_id"].unique()) == set(REPETITIONS)

# ------------------------------------------------------------
# 9. SOURCE-LEVEL COVERAGE REPORT
# ------------------------------------------------------------

source_coverage_records = []

for name, frame in prepared.items():

    source_coverage_records.append({
        "source": name,
        "available_records": len(frame),
        "expected_records": EXPECTED_RECORDS,
        "missing_records": EXPECTED_RECORDS - len(frame),
        "available_metrics": sum(
            column in frame.columns
            and frame[column].notna().any()
            for column in SOURCE_METRICS[name]
        ),
        "coverage_complete": len(frame) == EXPECTED_RECORDS
    })

source_coverage_df = pd.DataFrame(
    source_coverage_records
)

# ------------------------------------------------------------
# 10. SAVE ALIGNED DATA AND VALIDATION ARTIFACTS
# ------------------------------------------------------------

ALIGNED_RESULTS_PATH = (
    RESULTS_DIR / "aligned_experimental_conditions.csv"
)

SOURCE_COVERAGE_PATH = (
    VALIDATION_DIR / "aligned_source_coverage.csv"
)

aligned_df.to_csv(
    ALIGNED_RESULTS_PATH,
    index=False
)

source_coverage_df.to_csv(
    SOURCE_COVERAGE_PATH,
    index=False
)

# ------------------------------------------------------------
# 11. SECTION VALIDATION
# ------------------------------------------------------------

SECTION_06_VALIDATION = {
    "complete_experiment_grid": (
        len(aligned_df) == EXPECTED_RECORDS
    ),
    "unique_experimental_keys": (
        not aligned_df.duplicated(KEY_COLUMNS).any()
    ),
    "no_missing_experimental_keys": (
        aligned_df[KEY_COLUMNS].notna().all().all()
    ),
    "all_datasets_present": (
        set(aligned_df["dataset"].unique()) == set(DATASETS)
    ),
    "all_variants_present": (
        set(aligned_df["variant"].unique()) == set(VARIANTS)
    ),
    "all_repetitions_present": (
        set(aligned_df["repetition_id"].unique())
        == set(REPETITIONS)
    ),
    "aligned_results_saved": ALIGNED_RESULTS_PATH.is_file(),
    "source_coverage_saved": SOURCE_COVERAGE_PATH.is_file()
}

SECTION_06_STATUS = (
    "PASS"
    if all(SECTION_06_VALIDATION.values())
    else "FAIL"
)

# ------------------------------------------------------------
# 12. EXECUTION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 06 — ALIGN EXPERIMENTAL CONDITIONS")
print("=" * 75)

print("Expected experiments      :", EXPECTED_RECORDS)
print("Aligned records           :", len(aligned_df))
print(
    "Unique experimental keys :",
    not aligned_df.duplicated(KEY_COLUMNS).any()
)

print("-" * 75)

print("Fidelity source records   :", len(prepared["fidelity"]))
print("Utility source records    :", len(prepared["utility"]))
print("Privacy source records    :", len(prepared["privacy"]))
print("DP accounting records     :", len(prepared["dp"]))

print("-" * 75)

print(
    "Configured DP records    :",
    int(aligned_df["dp_configured"].sum())
)
print(
    "Formal DP verified       :",
    int(aligned_df["formal_dp_verified"].sum())
)

print("-" * 75)

print("Aligned results saved    :", ALIGNED_RESULTS_PATH)
print("Coverage report saved    :", SOURCE_COVERAGE_PATH)

print("-" * 75)

print("Validation checks        :", len(SECTION_06_VALIDATION))
print(
    "Passed checks            :",
    sum(SECTION_06_VALIDATION.values())
)
print(
    "Failed checks            :",
    len(SECTION_06_VALIDATION)
    - sum(SECTION_06_VALIDATION.values())
)
print("Section 06 status        :", SECTION_06_STATUS)

print("=" * 75)

SECTION 06 — ALIGN EXPERIMENTAL CONDITIONS
Expected experiments      : 60
Aligned records           : 60
Unique experimental keys : True
---------------------------------------------------------------------------
Fidelity source records   : 0
Utility source records    : 0
Privacy source records    : 0
DP accounting records     : 60
---------------------------------------------------------------------------
Configured DP records    : 30
Formal DP verified       : 0
---------------------------------------------------------------------------
Aligned results saved    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/aligned_experimental_conditions.csv
Coverage report saved    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/validation/aligned_source_coverage.csv
---------------------------------------------------------------------------
Validation checks        : 8
Passed checks            : 8
Failed checks            : 0
Section 06 stat

In [34]:
# ============================================================
# SECTION 07 — NORMALIZE METRICS
# SPP-GAN TRADE-OFF ANALYSIS
#
# 1. Fidelity: higher normalized score = better fidelity
# 2. Utility: higher normalized score = better utility
# 3. Empirical privacy: higher normalized score = lower attack risk
# 4. Formal DP: lower achieved epsilon = stronger protection
#
# Important:
# - Empirical privacy is separate from formal DP.
# - Only verified DP accounting records receive DP protection scores.
# - Missing evidence remains NaN; no artificial values are generated.
# - Original raw columns are preserved.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

# Target privacy budget from the SPP-GAN methodology.
# Do not depend on an undefined DP_CONFIG variable.
TARGET_EPSILON = 5.0

# Required experimental identifiers.
required_key_columns = list(KEY_COLUMNS)

# ------------------------------------------------------------
# 2. METRIC DEFINITIONS
# ------------------------------------------------------------

METRIC_DEFINITIONS = {

    # ---------------- FIDELITY ----------------

    "ks_similarity": ("fidelity", "maximize", 0, 1),
    "ks_statistic": ("fidelity", "minimize", 0, 1),
    "ks_distance": ("fidelity", "minimize", 0, 1),

    "wasserstein_distance": (
        "fidelity", "minimize", None, None
    ),

    "js_divergence": ("fidelity", "minimize", 0, 1),

    "mmd": ("fidelity", "minimize", None, None),

    "correlation_error": (
        "fidelity", "minimize", None, None
    ),

    "categorical_similarity": (
        "fidelity", "maximize", 0, 1
    ),

    "fidelity_score": ("fidelity", "maximize", 0, 1),

    # ---------------- UTILITY ----------------

    "downstream_utility": (
        "utility", "maximize", 0, 1
    ),

    "utility_score": ("utility", "maximize", 0, 1),

    "tstr_accuracy": ("utility", "maximize", 0, 1),

    "tstr_f1": ("utility", "maximize", 0, 1),

    "macro_f1": ("utility", "maximize", 0, 1),

    "weighted_f1": ("utility", "maximize", 0, 1),

    "f1": ("utility", "maximize", 0, 1),

    "accuracy": ("utility", "maximize", 0, 1),

    "auc": ("utility", "maximize", 0, 1),

    "roc_auc": ("utility", "maximize", 0, 1),

    "r2": ("utility", "maximize", None, 1),

    "rmse": ("utility", "minimize", None, None),

    "mae": ("utility", "minimize", None, None),

    # ---------------- EMPIRICAL PRIVACY ----------------
    # These are attack-risk measures, not formal DP guarantees.

    "membership_inference_auc": (
        "empirical_privacy", "minimize", 0.5, 1
    ),

    "mia_auc": (
        "empirical_privacy", "minimize", 0.5, 1
    ),

    "membership_inference_advantage": (
        "empirical_privacy", "minimize", 0, 1
    ),

    "mia_advantage": (
        "empirical_privacy", "minimize", 0, 1
    ),

    "reidentification_risk": (
        "empirical_privacy", "minimize", 0, 1
    ),

    "reidentification_rate": (
        "empirical_privacy", "minimize", 0, 1
    ),

    "disclosure_risk": (
        "empirical_privacy", "minimize", 0, 1
    ),

    "attribute_inference_accuracy": (
        "empirical_privacy", "minimize", 0, 1
    ),
}

# ------------------------------------------------------------
# 3. VALIDATE INPUT DATA
# ------------------------------------------------------------

if "aligned_df" not in globals():
    raise NameError(
        "aligned_df is unavailable. Run Section 06 first."
    )

if not isinstance(aligned_df, pd.DataFrame):
    raise TypeError("aligned_df must be a pandas DataFrame.")

missing_key_columns = [
    col for col in required_key_columns
    if col not in aligned_df.columns
]

if missing_key_columns:
    raise ValueError(
        f"Missing experimental key columns: {missing_key_columns}"
    )

if aligned_df.empty:
    raise ValueError(
        "aligned_df contains no experimental records."
    )

if aligned_df[required_key_columns].isna().any().any():
    raise ValueError(
        "Experimental key columns contain missing values."
    )

if aligned_df.duplicated(required_key_columns).any():
    raise ValueError(
        "Duplicate experimental keys found in aligned_df."
    )

# ------------------------------------------------------------
# 4. INITIALIZE OUTPUT
# ------------------------------------------------------------

# Preserve all original raw columns and experimental evidence.
normalized_df = aligned_df.copy()

normalization_records = []

# ------------------------------------------------------------
# 5. BOOLEAN NORMALIZATION HELPER
# ------------------------------------------------------------

def parse_boolean_series(series):

    """
    Safely convert Boolean-like values into True/False.

    Handles:
    - True / False
    - 1 / 0
    - "true" / "false"
    - "yes" / "no"
    - Missing values
    """

    true_values = {
        "true", "1", "yes", "y", "verified", "pass"
    }

    false_values = {
        "false", "0", "no", "n", "unverified", "fail", ""
    }

    def convert(value):

        if pd.isna(value):
            return False

        if isinstance(value, (bool, np.bool_)):
            return bool(value)

        if isinstance(value, (int, float, np.integer, np.floating)):
            return bool(value == 1)

        value = str(value).strip().lower()

        if value in true_values:
            return True

        if value in false_values:
            return False

        return False

    return series.apply(convert).astype(bool)


# ------------------------------------------------------------
# 6. NORMALIZE FIDELITY, UTILITY AND EMPIRICAL PRIVACY
# ------------------------------------------------------------

for metric, definition in METRIC_DEFINITIONS.items():

    domain, direction, lower, upper = definition

    # --------------------------------------------------------
    # Retrieve metric values
    # --------------------------------------------------------

    if metric not in aligned_df.columns:

        values = pd.Series(
            np.nan,
            index=aligned_df.index,
            dtype=float
        )

        source_status = "column_missing"

    else:

        values = pd.to_numeric(
            aligned_df[metric],
            errors="coerce"
        )

        values = values.replace(
            [np.inf, -np.inf],
            np.nan
        )

        source_status = "column_present"

    observed = values.dropna()

    output = pd.Series(
        np.nan,
        index=aligned_df.index,
        dtype=float
    )

    normalization_type = "unavailable"
    valid_bounds = 0

    # --------------------------------------------------------
    # Normalize available values
    # --------------------------------------------------------

    if not observed.empty:

        if lower is not None and upper is not None:

            low = float(lower)
            high = float(upper)

            normalization_type = "fixed_bounds"

        else:

            low = (
                float(lower)
                if lower is not None
                else float(observed.min())
            )

            high = (
                float(upper)
                if upper is not None
                else float(observed.max())
            )

            normalization_type = "observed_bounds"

        if high > low:

            valid = (
                values.notna()
                & values.ge(low)
                & values.le(high)
            )

            valid_bounds = int(valid.sum())

            if direction == "maximize":

                output.loc[valid] = (
                    (values.loc[valid] - low)
                    / (high - low)
                )

            elif direction == "minimize":

                output.loc[valid] = (
                    (high - values.loc[valid])
                    / (high - low)
                )

            else:

                raise ValueError(
                    f"Unsupported metric direction: {direction}"
                )

        else:

            # A constant metric cannot distinguish alternatives.
            normalization_type = "constant_uninformative"

    # --------------------------------------------------------
    # Save normalized metric
    # --------------------------------------------------------

    normalized_df[f"norm_{metric}"] = output

    normalization_records.append({

        "metric": metric,
        "domain": domain,
        "direction": direction,

        "source_status": source_status,

        "normalization": normalization_type,

        "source_records": int(values.notna().sum()),

        "valid_records": int(output.notna().sum()),

        "excluded_out_of_bounds": int(
            values.notna().sum() - valid_bounds
        ),

        "fixed_lower": lower,
        "fixed_upper": upper

    })


# ------------------------------------------------------------
# 7. FORMAL DIFFERENTIAL PRIVACY NORMALIZATION
# ------------------------------------------------------------
#
# Lower achieved epsilon = stronger formal DP protection.
#
# Normalized protection:
#
#   DP_Protection = 1 - min(achieved_epsilon / target_epsilon, 1)
#
# This is calculated only for verified formal accounting records.
#
# It is not an empirical attack-risk score.
# ------------------------------------------------------------

epsilon_source = None

if "achieved_epsilon" in aligned_df.columns:

    epsilon_source = "achieved_epsilon"

elif "epsilon" in aligned_df.columns:

    epsilon_source = "epsilon"


# ------------------------------------------------------------
# Retrieve achieved epsilon
# ------------------------------------------------------------

if epsilon_source is not None:

    epsilon_values = pd.to_numeric(
        aligned_df[epsilon_source],
        errors="coerce"
    )

    epsilon_values = epsilon_values.replace(
        [np.inf, -np.inf],
        np.nan
    )

else:

    epsilon_values = pd.Series(
        np.nan,
        index=aligned_df.index,
        dtype=float
    )


# ------------------------------------------------------------
# Determine formal DP verification
# ------------------------------------------------------------

if "formal_dp_verified" in aligned_df.columns:

    formal_verified = parse_boolean_series(
        aligned_df["formal_dp_verified"]
    )

elif "valid_accounting_evidence" in aligned_df.columns:

    formal_verified = parse_boolean_series(
        aligned_df["valid_accounting_evidence"]
    )

else:

    formal_verified = pd.Series(
        False,
        index=aligned_df.index
    )


# ------------------------------------------------------------
# Additional safeguards
# ------------------------------------------------------------

if "dp_configured" in aligned_df.columns:

    dp_configured = parse_boolean_series(
        aligned_df["dp_configured"]
    )

else:

    dp_configured = pd.Series(
        False,
        index=aligned_df.index
    )


if "dp_applied" in aligned_df.columns:

    dp_applied = parse_boolean_series(
        aligned_df["dp_applied"]
    )

else:

    dp_applied = pd.Series(
        False,
        index=aligned_df.index
    )


# If DP configuration/application fields exist, require them.
# A configured DP run alone is not proof that DP was applied.

dp_execution_valid = (
    dp_configured & dp_applied
)

# If these fields are absent, no record is considered verified.
valid_formal_dp = (
    formal_verified
    & dp_execution_valid
    & epsilon_values.notna()
    & epsilon_values.gt(0)
)


# ------------------------------------------------------------
# Validate delta where available
# ------------------------------------------------------------

if "delta" in aligned_df.columns:

    delta_values = pd.to_numeric(
        aligned_df["delta"],
        errors="coerce"
    )

    valid_delta = (
        delta_values.notna()
        & delta_values.gt(0)
        & delta_values.lt(1)
    )

    valid_formal_dp &= valid_delta


# ------------------------------------------------------------
# Normalize formal DP protection
# ------------------------------------------------------------

normalized_dp_protection = pd.Series(
    np.nan,
    index=aligned_df.index,
    dtype=float
)

normalized_dp_protection.loc[valid_formal_dp] = (

    1.0
    -
    (
        epsilon_values.loc[valid_formal_dp]
        / TARGET_EPSILON
    ).clip(lower=0, upper=1)

)

normalized_df["achieved_epsilon_raw"] = epsilon_values

normalized_df["formal_dp_verified_for_analysis"] = (
    valid_formal_dp
)

normalized_df["norm_dp_privacy_protection"] = (
    normalized_dp_protection
)


# ------------------------------------------------------------
# Register formal DP normalization
# ------------------------------------------------------------

normalization_records.append({

    "metric": "achieved_epsilon",

    "domain": "formal_dp_privacy",

    "direction": "minimize",

    "source_status": (
        "column_present"
        if epsilon_source is not None
        else "column_missing"
    ),

    "normalization": "target_epsilon_based",

    "source_records": int(
        epsilon_values.notna().sum()
    ),

    "valid_records": int(
        valid_formal_dp.sum()
    ),

    "excluded_out_of_bounds": 0,

    "fixed_lower": 0,

    "fixed_upper": TARGET_EPSILON

})


# ------------------------------------------------------------
# 8. CREATE NORMALIZATION REGISTRY
# ------------------------------------------------------------

normalization_metadata_df = pd.DataFrame(
    normalization_records
)


# ------------------------------------------------------------
# 9. VALIDATION
# ------------------------------------------------------------

normalized_metric_columns = [

    col for col in normalized_df.columns
    if col.startswith("norm_")

]

valid_measurements = int(

    normalized_df[normalized_metric_columns]
    .notna()
    .sum()
    .sum()

)

formal_dp_verified_count = int(
    normalized_df[
        "formal_dp_verified_for_analysis"
    ].sum()
)

# Check record preservation.
assert len(normalized_df) == len(aligned_df), (
    "Normalized row count does not match aligned data."
)

# Check experimental key uniqueness.
assert not normalized_df.duplicated(
    required_key_columns
).any(), (
    "Duplicate experimental keys after normalization."
)

# Check normalized values are within [0, 1].
for col in normalized_metric_columns:

    valid_values = normalized_df[col].dropna()

    if not valid_values.empty:

        assert valid_values.between(0, 1).all(), (
            f"Normalized values outside [0, 1]: {col}"
        )

# Check formal DP scores are assigned only to verified records.
assert normalized_df.loc[
    ~valid_formal_dp,
    "norm_dp_privacy_protection"
].isna().all(), (
    "Unverified records received a formal DP protection score."
)


# ------------------------------------------------------------
# 10. SAVE NORMALIZED RESULTS
# ------------------------------------------------------------

normalized_output_path = (
    RESULTS_DIR / "normalized_metrics.csv"
)

normalization_registry_path = (
    METADATA_DIR / "normalization_registry.csv"
)

normalized_df.to_csv(
    normalized_output_path,
    index=False
)

normalization_metadata_df.to_csv(
    normalization_registry_path,
    index=False
)


# ------------------------------------------------------------
# 11. SAVE FORMAL DP NORMALIZATION SUMMARY
# ------------------------------------------------------------

dp_normalization_summary = pd.DataFrame([{

    "expected_experiments": len(aligned_df),

    "target_epsilon": TARGET_EPSILON,

    "epsilon_source_column": epsilon_source,

    "records_with_epsilon": int(
        epsilon_values.notna().sum()
    ),

    "configured_dp_records": int(
        dp_configured.sum()
    ),

    "dp_applied_records": int(
        dp_applied.sum()
    ),

    "formally_verified_records": formal_dp_verified_count,

    "records_with_normalized_dp_protection": int(
        normalized_dp_protection.notna().sum()
    ),

    "formal_dp_status": (
        "VERIFIED_EVIDENCE_AVAILABLE"
        if formal_dp_verified_count > 0
        else "NO_VERIFIED_FORMAL_DP_EVIDENCE"
    )

}])

dp_normalization_summary_path = (
    METADATA_DIR / "dp_normalization_summary.csv"
)

dp_normalization_summary.to_csv(
    dp_normalization_summary_path,
    index=False
)


# ------------------------------------------------------------
# 12. SAVE SECTION VALIDATION REPORT
# ------------------------------------------------------------

section_validation = [

    {
        "check": "aligned_records_preserved",
        "status": len(normalized_df) == len(aligned_df)
    },

    {
        "check": "experimental_keys_unique",
        "status": not normalized_df.duplicated(
            required_key_columns
        ).any()
    },

    {
        "check": "normalized_values_within_0_1",
        "status": all(
            normalized_df[col].dropna().between(0, 1).all()
            for col in normalized_metric_columns
        )
    },

    {
        "check": "unverified_dp_scores_remain_missing",
        "status": normalized_df.loc[
            ~valid_formal_dp,
            "norm_dp_privacy_protection"
        ].isna().all()
    },

    {
        "check": "normalization_registry_created",
        "status": not normalization_metadata_df.empty
    }

]

section_validation_df = pd.DataFrame(
    section_validation
)

section_validation_path = (
    METADATA_DIR / "section_07_validation.csv"
)

section_validation_df.to_csv(
    section_validation_path,
    index=False
)

passed_checks = int(
    section_validation_df["status"].sum()
)

failed_checks = len(section_validation_df) - passed_checks

section_status = (
    "PASS"
    if failed_checks == 0
    else "FAILED"
)


# ------------------------------------------------------------
# 13. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 07 — NORMALIZE METRICS")
print("=" * 75)

print("Aligned records               :", len(aligned_df))
print("Normalized records            :", len(normalized_df))

print("Defined empirical metrics     :", len(METRIC_DEFINITIONS))
print("Available normalized columns  :", len(normalized_metric_columns))

print("Valid normalized measurements :", valid_measurements)

print("-" * 75)

print("Target epsilon                 :", TARGET_EPSILON)

print(
    "Records with achieved epsilon :",
    int(epsilon_values.notna().sum())
)

print("Configured DP records         :", int(dp_configured.sum()))
print("DP applied records            :", int(dp_applied.sum()))

print("Formal DP verified records    :", formal_dp_verified_count)

print(
    "Normalized DP protection     :",
    int(normalized_dp_protection.notna().sum())
)

print("-" * 75)

print("Normalized results saved      :", normalized_output_path)
print("Normalization registry saved  :", normalization_registry_path)
print("DP normalization summary      :", dp_normalization_summary_path)
print("Section validation saved      :", section_validation_path)

print("-" * 75)

print("Validation checks              :", len(section_validation_df))
print("Passed checks                  :", passed_checks)
print("Failed checks                  :", failed_checks)

print("Section 07 status              :", section_status)

print("=" * 75)

SECTION 07 — NORMALIZE METRICS
Aligned records               : 60
Normalized records            : 60
Defined empirical metrics     : 30
Available normalized columns  : 31
Valid normalized measurements : 0
---------------------------------------------------------------------------
Target epsilon                 : 5.0
Records with achieved epsilon : 0
Configured DP records         : 30
DP applied records            : 30
Formal DP verified records    : 0
Normalized DP protection     : 0
---------------------------------------------------------------------------
Normalized results saved      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/normalized_metrics.csv
Normalization registry saved  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/metadata/normalization_registry.csv
DP normalization summary      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/metadata/dp_normalization_summary.csv
Section validation saved

In [36]:
# ============================================================
# SECTION 08 — DEFINE TRADE-OFF MEASURES
# SPP-GAN PRIVACY-FIDELITY-UTILITY ANALYSIS
#
# Primary dimensions:
#   1. Formal DP privacy protection: achieved epsilon
#   2. Fidelity: statistical similarity
#   3. Utility: downstream TSTR performance
#
# Empirical privacy attacks are maintained separately.
# Missing measurements are never imputed.
# ============================================================

import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TARGET_EPSILON = 5.0

TRADEOFF_WEIGHTS = {
    "privacy": 1 / 3,
    "fidelity": 1 / 3,
    "utility": 1 / 3
}

# ------------------------------------------------------------
# 2. METRIC PRIORITY
# ------------------------------------------------------------

METRIC_PRIORITY = {

    # ---------------- FIDELITY ----------------

    "fidelity": [
        "fidelity_score",
        "ks_similarity",
        "categorical_similarity",
        "ks_statistic",
        "ks_distance",
        "wasserstein_distance",
        "js_divergence",
        "mmd",
        "correlation_error"
    ],

    # ---------------- UTILITY ----------------

    "utility": [
        "downstream_utility",
        "utility_score",
        "tstr_f1",
        "macro_f1",
        "weighted_f1",
        "f1",
        "accuracy",
        "auc",
        "roc_auc",
        "r2",
        "rmse",
        "mae"
    ],

    # ---------------- EMPIRICAL PRIVACY ----------------
    # Secondary analysis only; not the formal DP axis.

    "empirical_privacy": [
        "membership_inference_advantage",
        "mia_advantage",
        "membership_inference_auc",
        "mia_auc",
        "reidentification_risk",
        "reidentification_rate",
        "disclosure_risk",
        "attribute_inference_accuracy"
    ]
}

# ------------------------------------------------------------
# 3. VALIDATE INPUTS
# ------------------------------------------------------------

required_objects = [
    "aligned_df",
    "normalized_df",
    "KEY_COLUMNS",
    "EXPECTED_RECORDS",
    "RESULTS_DIR",
    "METADATA_DIR"
]

for object_name in required_objects:

    if object_name not in globals():

        raise NameError(
            f"{object_name} is unavailable. "
            "Run the preceding Notebook 18 sections first."
        )

if len(aligned_df) != EXPECTED_RECORDS:

    raise ValueError(
        f"Expected {EXPECTED_RECORDS} aligned records, "
        f"but found {len(aligned_df)}."
    )

if len(normalized_df) != EXPECTED_RECORDS:

    raise ValueError(
        "Normalized dataset does not match expected records."
    )

if aligned_df.duplicated(KEY_COLUMNS).any():

    raise ValueError(
        "Duplicate experimental keys detected."
    )

# ------------------------------------------------------------
# 4. METRIC SELECTION FUNCTION
# ------------------------------------------------------------

def select_available_metric(domain, priority_registry):

    """
    Select the first metric with at least one valid normalized value.

    Returns None if no actual measurement is available.
    """

    for metric in priority_registry[domain]:

        column = f"norm_{metric}"

        if column not in normalized_df.columns:
            continue

        values = pd.to_numeric(
            normalized_df[column],
            errors="coerce"
        )

        values = values.replace(
            [np.inf, -np.inf],
            np.nan
        )

        if values.notna().any():

            return metric

    return None


# ------------------------------------------------------------
# 5. SELECT FIDELITY AND UTILITY METRICS
# ------------------------------------------------------------

SELECTED_METRICS = {

    "fidelity": select_available_metric(
        "fidelity",
        METRIC_PRIORITY
    ),

    "utility": select_available_metric(
        "utility",
        METRIC_PRIORITY
    ),

    "empirical_privacy": select_available_metric(
        "empirical_privacy",
        METRIC_PRIORITY
    )

}

# ------------------------------------------------------------
# 6. SELECT FORMAL DP PRIVACY MEASURE
# ------------------------------------------------------------

FORMAL_PRIVACY_METRIC = "norm_dp_privacy_protection"

if (
    FORMAL_PRIVACY_METRIC in normalized_df.columns
    and normalized_df[FORMAL_PRIVACY_METRIC].notna().any()
):

    formal_privacy_available = True

else:

    formal_privacy_available = False


# ------------------------------------------------------------
# 7. INITIALIZE TRADE-OFF DATAFRAME
# ------------------------------------------------------------

tradeoff_columns = list(KEY_COLUMNS)

if "dp_configured" in aligned_df.columns:
    tradeoff_columns.append("dp_configured")

if "dp_applied" in aligned_df.columns:
    tradeoff_columns.append("dp_applied")

tradeoff_df = aligned_df[
    tradeoff_columns
].copy()

# ------------------------------------------------------------
# 8. ADD FORMAL PRIVACY SCORE
# ------------------------------------------------------------

if formal_privacy_available:

    tradeoff_df["privacy_score"] = (
        normalized_df[FORMAL_PRIVACY_METRIC]
        .values
    )

else:

    tradeoff_df["privacy_score"] = np.nan

# Preserve raw achieved epsilon for interpretation.
if "achieved_epsilon_raw" in normalized_df.columns:

    tradeoff_df["achieved_epsilon"] = (
        normalized_df["achieved_epsilon_raw"]
        .values
    )

else:

    tradeoff_df["achieved_epsilon"] = np.nan

# Preserve formal verification status.
if "formal_dp_verified_for_analysis" in normalized_df.columns:

    tradeoff_df["formal_dp_verified"] = (
        normalized_df["formal_dp_verified_for_analysis"]
        .values
    )

else:

    tradeoff_df["formal_dp_verified"] = False


# ------------------------------------------------------------
# 9. ADD FIDELITY AND UTILITY SCORES
# ------------------------------------------------------------

for domain in ["fidelity", "utility"]:

    metric = SELECTED_METRICS[domain]

    if metric is not None:

        normalized_column = f"norm_{metric}"

        tradeoff_df[f"{domain}_score"] = (
            normalized_df[normalized_column].values
        )

    else:

        tradeoff_df[f"{domain}_score"] = np.nan


# ------------------------------------------------------------
# 10. ADD EMPIRICAL PRIVACY SEPARATELY
# ------------------------------------------------------------

empirical_metric = SELECTED_METRICS["empirical_privacy"]

if empirical_metric is not None:

    tradeoff_df["empirical_privacy_score"] = (
        normalized_df[
            f"norm_{empirical_metric}"
        ].values
    )

else:

    tradeoff_df["empirical_privacy_score"] = np.nan


# ------------------------------------------------------------
# 11. DEFINE COMPLETE THREE-WAY RECORDS
# ------------------------------------------------------------

SCORE_COLUMNS = [
    "privacy_score",
    "fidelity_score",
    "utility_score"
]

tradeoff_df["complete_three_way"] = (
    tradeoff_df[SCORE_COLUMNS]
    .notna()
    .all(axis=1)
)

tradeoff_df["tradeoff_score"] = np.nan

valid_three_way = tradeoff_df["complete_three_way"]

if valid_three_way.any():

    tradeoff_df.loc[
        valid_three_way,
        "tradeoff_score"
    ] = (

        TRADEOFF_WEIGHTS["privacy"]
        * tradeoff_df.loc[
            valid_three_way,
            "privacy_score"
        ]

        +

        TRADEOFF_WEIGHTS["fidelity"]
        * tradeoff_df.loc[
            valid_three_way,
            "fidelity_score"
        ]

        +

        TRADEOFF_WEIGHTS["utility"]
        * tradeoff_df.loc[
            valid_three_way,
            "utility_score"
        ]

    )


# ------------------------------------------------------------
# 12. DEFINE PAIRWISE ANALYSIS AVAILABILITY
# ------------------------------------------------------------

tradeoff_df["privacy_fidelity_available"] = (
    tradeoff_df[
        ["privacy_score", "fidelity_score"]
    ].notna().all(axis=1)
)

tradeoff_df["privacy_utility_available"] = (
    tradeoff_df[
        ["privacy_score", "utility_score"]
    ].notna().all(axis=1)
)

tradeoff_df["fidelity_utility_available"] = (
    tradeoff_df[
        ["fidelity_score", "utility_score"]
    ].notna().all(axis=1)
)


# ------------------------------------------------------------
# 13. TRADE-OFF CONFIGURATION
# ------------------------------------------------------------

tradeoff_config = {

    "section": "Section 08 — Define Trade-off Measures",

    "expected_experiments": int(EXPECTED_RECORDS),

    "target_epsilon": TARGET_EPSILON,

    "primary_tradeoff_dimensions": {

        "privacy": {
            "metric": "achieved_epsilon",
            "normalized_metric": FORMAL_PRIVACY_METRIC,
            "direction": "lower_epsilon_is_better",
            "formal_verification_required": True
        },

        "fidelity": {
            "selected_metric": SELECTED_METRICS["fidelity"],
            "direction": "higher_normalized_score_is_better"
        },

        "utility": {
            "selected_metric": SELECTED_METRICS["utility"],
            "direction": "higher_normalized_score_is_better"
        }

    },

    "secondary_empirical_privacy": {
        "selected_metric": empirical_metric,
        "interpretation": (
            "Empirical attack-risk evidence; "
            "not a formal differential privacy guarantee."
        )
    },

    "weights": TRADEOFF_WEIGHTS,

    "score_direction": "higher_is_better",

    "missing_data_policy": "no_imputation",

    "composite_score_policy": (
        "Calculate only when all three primary dimensions "
        "are available for the same experimental record."
    )

}


# ------------------------------------------------------------
# 14. VALIDATION
# ------------------------------------------------------------

assert len(tradeoff_df) == EXPECTED_RECORDS

assert not tradeoff_df.duplicated(
    KEY_COLUMNS
).any()

assert tradeoff_df["complete_three_way"].equals(
    tradeoff_df[SCORE_COLUMNS].notna().all(axis=1)
)

assert tradeoff_df.loc[
    ~tradeoff_df["complete_three_way"],
    "tradeoff_score"
].isna().all()

# ------------------------------------------------------------
# 15. SAVE OUTPUTS
# ------------------------------------------------------------

tradeoff_output_path = (
    RESULTS_DIR / "tradeoff_measures.csv"
)

tradeoff_config_path = (
    METADATA_DIR / "tradeoff_configuration.json"
)

tradeoff_coverage_path = (
    METADATA_DIR / "tradeoff_measure_coverage.csv"
)

tradeoff_df.to_csv(
    tradeoff_output_path,
    index=False
)

with open(
    tradeoff_config_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        tradeoff_config,
        file,
        indent=2
    )


# ------------------------------------------------------------
# 16. SAVE COVERAGE REPORT
# ------------------------------------------------------------

coverage_records = [

    {
        "dimension": "formal_privacy",
        "selected_metric": "achieved_epsilon",
        "available_records": int(
            tradeoff_df["privacy_score"].notna().sum()
        )
    },

    {
        "dimension": "fidelity",
        "selected_metric": SELECTED_METRICS["fidelity"],
        "available_records": int(
            tradeoff_df["fidelity_score"].notna().sum()
        )
    },

    {
        "dimension": "utility",
        "selected_metric": SELECTED_METRICS["utility"],
        "available_records": int(
            tradeoff_df["utility_score"].notna().sum()
        )
    },

    {
        "dimension": "empirical_privacy",
        "selected_metric": empirical_metric,
        "available_records": int(
            tradeoff_df[
                "empirical_privacy_score"
            ].notna().sum()
        )
    },

    {
        "dimension": "complete_three_way",
        "selected_metric": "privacy + fidelity + utility",
        "available_records": int(
            tradeoff_df["complete_three_way"].sum()
        )
    }

]

tradeoff_coverage_df = pd.DataFrame(
    coverage_records
)

tradeoff_coverage_df.to_csv(
    tradeoff_coverage_path,
    index=False
)


# ------------------------------------------------------------
# 17. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 08 — DEFINE TRADE-OFF MEASURES")
print("=" * 75)

print("Expected records              :", EXPECTED_RECORDS)
print("Trade-off records             :", len(tradeoff_df))

print("-" * 75)

print("Selected fidelity metric      :", SELECTED_METRICS["fidelity"])
print("Selected utility metric       :", SELECTED_METRICS["utility"])
print("Selected empirical privacy    :", empirical_metric)

print(
    "Formal DP privacy available   :",
    formal_privacy_available
)

print("-" * 75)

print(
    "Formal privacy records       :",
    int(tradeoff_df["privacy_score"].notna().sum())
)

print(
    "Fidelity records              :",
    int(tradeoff_df["fidelity_score"].notna().sum())
)

print(
    "Utility records               :",
    int(tradeoff_df["utility_score"].notna().sum())
)

print(
    "Empirical privacy records     :",
    int(tradeoff_df["empirical_privacy_score"].notna().sum())
)

print(
    "Complete three-way records   :",
    int(tradeoff_df["complete_three_way"].sum())
)

print("-" * 75)

print("Trade-off results saved       :", tradeoff_output_path)
print("Configuration saved           :", tradeoff_config_path)
print("Coverage report saved         :", tradeoff_coverage_path)

print("-" * 75)

print("Section 08 status             : PASS")
print("=" * 75)

SECTION 08 — DEFINE TRADE-OFF MEASURES
Expected records              : 60
Trade-off records             : 60
---------------------------------------------------------------------------
Selected fidelity metric      : None
Selected utility metric       : None
Selected empirical privacy    : None
Formal DP privacy available   : False
---------------------------------------------------------------------------
Formal privacy records       : 0
Fidelity records              : 0
Utility records               : 0
Empirical privacy records     : 0
Complete three-way records   : 0
---------------------------------------------------------------------------
Trade-off results saved       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/tradeoff_measures.csv
Configuration saved           : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/metadata/tradeoff_configuration.json
Coverage report saved         : /content/drive/MyDrive/SPP_GAN_Research/res

In [37]:
# ============================================================
# SECTION 09 — PRIVACY VS FIDELITY ANALYSIS
# ============================================================

def pairwise_analysis(frame, x_column, y_column, name):

    result = frame[
        KEY_COLUMNS + [x_column, y_column]
    ].dropna().copy()

    result = result.rename(columns={
        x_column: "x_score",
        y_column: "y_score"
    })

    result["analysis"] = name

    result.to_csv(
        RESULTS_DIR / f"{name}_results.csv",
        index=False
    )

    summary = []

    for dataset in DATASETS:
        subset = result[result["dataset"] == dataset]

        correlation = (
            subset["x_score"].corr(subset["y_score"])
            if len(subset) >= 3
            and subset["x_score"].nunique() > 1
            and subset["y_score"].nunique() > 1
            else np.nan
        )

        summary.append({
            "dataset": dataset,
            "analysis": name,
            "valid_records": len(subset),
            "pearson_correlation": correlation
        })

    summary_df = pd.DataFrame(summary)

    summary_df.to_csv(
        TABLES_DIR / f"{name}_summary.csv",
        index=False
    )

    return result, summary_df


privacy_fidelity_df, privacy_fidelity_summary = pairwise_analysis(
    tradeoff_df,
    "privacy_score",
    "fidelity_score",
    "privacy_vs_fidelity"
)

print("Valid paired records :", len(privacy_fidelity_df))
print(privacy_fidelity_summary.to_string(index=False))

Valid paired records : 0
       dataset            analysis  valid_records  pearson_correlation
  adult_income privacy_vs_fidelity              0                  NaN
bank_marketing privacy_vs_fidelity              0                  NaN
diabetes_130us privacy_vs_fidelity              0                  NaN


In [38]:
# ============================================================
# SECTION 10 — PRIVACY VS UTILITY ANALYSIS
# ============================================================

privacy_utility_df, privacy_utility_summary = pairwise_analysis(
    tradeoff_df,
    "privacy_score",
    "utility_score",
    "privacy_vs_utility"
)

print("Valid paired records :", len(privacy_utility_df))
print(privacy_utility_summary.to_string(index=False))

Valid paired records : 0
       dataset           analysis  valid_records  pearson_correlation
  adult_income privacy_vs_utility              0                  NaN
bank_marketing privacy_vs_utility              0                  NaN
diabetes_130us privacy_vs_utility              0                  NaN


In [39]:
# ============================================================
# SECTION 11 — FIDELITY VS UTILITY ANALYSIS
# ============================================================

fidelity_utility_df, fidelity_utility_summary = pairwise_analysis(
    tradeoff_df,
    "fidelity_score",
    "utility_score",
    "fidelity_vs_utility"
)

print("Valid paired records :", len(fidelity_utility_df))
print(fidelity_utility_summary.to_string(index=False))

Valid paired records : 0
       dataset            analysis  valid_records  pearson_correlation
  adult_income fidelity_vs_utility              0                  NaN
bank_marketing fidelity_vs_utility              0                  NaN
diabetes_130us fidelity_vs_utility              0                  NaN


In [40]:
# ============================================================
# SECTION 12 — THREE-WAY TRADE-OFF ANALYSIS
# ============================================================

three_way_df = tradeoff_df.loc[
    tradeoff_df["complete_three_way"]
].copy()

three_way_summary = []

for dataset in DATASETS:
    for variant in VARIANTS:

        subset = three_way_df.loc[
            (three_way_df["dataset"] == dataset)
            & (three_way_df["variant"] == variant)
        ]

        record = {
            "dataset": dataset,
            "variant": variant,
            "valid_runs": len(subset)
        }

        for metric in SCORE_COLUMNS + ["tradeoff_score"]:
            record[f"{metric}_mean"] = (
                subset[metric].mean()
                if len(subset) else np.nan
            )

            record[f"{metric}_std"] = (
                subset[metric].std()
                if len(subset) > 1 else np.nan
            )

        three_way_summary.append(record)

three_way_summary_df = pd.DataFrame(three_way_summary)

three_way_df.to_csv(
    RESULTS_DIR / "three_way_tradeoff_results.csv",
    index=False
)

three_way_summary_df.to_csv(
    TABLES_DIR / "three_way_tradeoff_summary.csv",
    index=False
)

print("Complete three-way records :", len(three_way_df))
print("Dataset × variant rows     :", len(three_way_summary_df))

Complete three-way records : 0
Dataset × variant rows     : 12


In [42]:
# ============================================================
# SECTION 13 — PARETO FRONTIER
# SPP-GAN PRIVACY-FIDELITY-UTILITY ANALYSIS
#
# Objectives:
#   1. Maximize formal DP privacy protection
#   2. Maximize fidelity
#   3. Maximize utility
#
# Only complete, finite three-way observations are evaluated.
# Missing evidence is never imputed.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PARETO_OBJECTIVES = [
    "privacy_score",
    "fidelity_score",
    "utility_score"
]

# ------------------------------------------------------------
# 2. VALIDATE INPUTS
# ------------------------------------------------------------

if "three_way_df" not in globals():
    raise NameError(
        "three_way_df is unavailable. Run Section 12 first."
    )

if "DATASETS" not in globals():
    raise NameError("DATASETS is unavailable.")

if "RESULTS_DIR" not in globals():
    raise NameError("RESULTS_DIR is unavailable.")

missing_objectives = [
    col for col in PARETO_OBJECTIVES
    if col not in tradeoff_df.columns
]

if missing_objectives:
    raise ValueError(
        f"Missing Pareto objective columns: {missing_objectives}"
    )

# ------------------------------------------------------------
# 3. PARETO FRONTIER FUNCTION
# ------------------------------------------------------------

def calculate_pareto_frontier(
    frame,
    objectives,
    maximize=True
):
    """
    Identify nondominated observations.

    maximize=True:
        Higher values are better for every objective.

    A solution is dominated when another solution is:
        - At least as good on every objective.
        - Strictly better on at least one objective.
    """

    result = frame.copy()

    result["pareto_optimal"] = False

    if result.empty:
        result["pareto_status"] = "NO_VALID_SOLUTIONS"
        return result

    missing_columns = [
        col for col in objectives
        if col not in result.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing Pareto columns: {missing_columns}"
        )

    values = result[objectives].apply(
        pd.to_numeric,
        errors="coerce"
    ).to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            "Pareto objectives contain NaN or infinite values. "
            "Filter incomplete records before calling this function."
        )

    if maximize:
        comparison_values = values
    else:
        comparison_values = -values

    pareto = np.ones(
        len(comparison_values),
        dtype=bool
    )

    for i in range(len(comparison_values)):

        dominated = (
            np.all(
                comparison_values >= comparison_values[i],
                axis=1
            )
            &
            np.any(
                comparison_values > comparison_values[i],
                axis=1
            )
        )

        # Exclude the observation itself.
        dominated[i] = False

        pareto[i] = not dominated.any()

    result["pareto_optimal"] = pareto

    result["pareto_status"] = np.where(
        pareto,
        "PARETO_OPTIMAL",
        "DOMINATED"
    )

    return result


# ------------------------------------------------------------
# 4. PREPARE COMPLETE THREE-WAY OBSERVATIONS
# ------------------------------------------------------------

pareto_input_df = three_way_df.copy()

# Retain only complete objective records.
pareto_input_df = pareto_input_df.dropna(
    subset=PARETO_OBJECTIVES
).copy()

# Ensure objectives are numeric.
for col in PARETO_OBJECTIVES:

    pareto_input_df[col] = pd.to_numeric(
        pareto_input_df[col],
        errors="coerce"
    )

# Remove non-finite objective records.
finite_mask = np.isfinite(
    pareto_input_df[PARETO_OBJECTIVES].to_numpy(
        dtype=float
    )
).all(axis=1)

pareto_input_df = pareto_input_df.loc[
    finite_mask
].copy()

# ------------------------------------------------------------
# 5. CALCULATE DATASET-LEVEL PARETO FRONTIERS
# ------------------------------------------------------------

pareto_parts = []

dataset_pareto_summary = []

for dataset in DATASETS:

    subset = pareto_input_df.loc[
        pareto_input_df["dataset"] == dataset
    ].copy()

    evaluated_subset = calculate_pareto_frontier(
        subset,
        PARETO_OBJECTIVES,
        maximize=True
    )

    pareto_parts.append(evaluated_subset)

    dataset_pareto_summary.append({

        "dataset": dataset,

        "evaluated_solutions": len(evaluated_subset),

        "pareto_optimal_solutions": int(
            evaluated_subset["pareto_optimal"].sum()
        ),

        "dominated_solutions": int(
            (
                evaluated_subset["pareto_status"]
                == "DOMINATED"
            ).sum()
        ),

        "status": (
            "EVALUATED"
            if len(evaluated_subset) > 0
            else "NO_VALID_SOLUTIONS"
        )

    })


# ------------------------------------------------------------
# 6. COMBINE RESULTS
# ------------------------------------------------------------

if pareto_parts:

    pareto_df = pd.concat(
        pareto_parts,
        ignore_index=True
    )

else:

    pareto_df = pareto_input_df.copy()

    pareto_df["pareto_optimal"] = False

    pareto_df["pareto_status"] = (
        "NO_VALID_SOLUTIONS"
    )


pareto_frontier_df = pareto_df.loc[
    pareto_df["pareto_optimal"]
].copy()

pareto_summary_df = pd.DataFrame(
    dataset_pareto_summary
)

# ------------------------------------------------------------
# 7. SAVE PARETO RESULTS
# ------------------------------------------------------------

pareto_evaluation_path = (
    RESULTS_DIR / "pareto_evaluation.csv"
)

pareto_frontier_path = (
    RESULTS_DIR / "pareto_frontier.csv"
)

pareto_summary_path = (
    TABLES_DIR / "pareto_frontier_summary.csv"
)

pareto_df.to_csv(
    pareto_evaluation_path,
    index=False
)

pareto_frontier_df.to_csv(
    pareto_frontier_path,
    index=False
)

pareto_summary_df.to_csv(
    pareto_summary_path,
    index=False
)

# ------------------------------------------------------------
# 8. VALIDATION
# ------------------------------------------------------------

validation_checks = [

    {
        "check": "pareto_input_has_complete_objectives",
        "status": (
            pareto_input_df[PARETO_OBJECTIVES]
            .notna()
            .all()
            .all()
        )
    },

    {
        "check": "pareto_results_preserve_input_count",
        "status": (
            len(pareto_df) == len(pareto_input_df)
        )
    },

    {
        "check": "pareto_frontier_contains_only_optimal_records",
        "status": (
            pareto_frontier_df["pareto_optimal"].all()
            if len(pareto_frontier_df)
            else True
        )
    },

    {
        "check": "dataset_summary_created",
        "status": (
            len(pareto_summary_df) == len(DATASETS)
        )
    },

    {
        "check": "pareto_output_files_saved",
        "status": (
            pareto_evaluation_path.exists()
            and pareto_frontier_path.exists()
            and pareto_summary_path.exists()
        )
    }

]

pareto_validation_df = pd.DataFrame(
    validation_checks
)

pareto_validation_path = (
    METADATA_DIR / "section_13_validation.csv"
)

pareto_validation_df.to_csv(
    pareto_validation_path,
    index=False
)

passed_checks = int(
    pareto_validation_df["status"].sum()
)

failed_checks = (
    len(pareto_validation_df) - passed_checks
)

section_status = (
    "PASS"
    if failed_checks == 0
    else "FAILED"
)

# ------------------------------------------------------------
# 9. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 13 — PARETO FRONTIER")
print("=" * 75)

print("Expected experiments       :", EXPECTED_RECORDS)

print("Complete input solutions   :", len(pareto_input_df))

print("Evaluated solutions        :", len(pareto_df))

print(
    "Pareto-optimal solutions   :",
    len(pareto_frontier_df)
)

print(
    "Datasets evaluated         :",
    int(
        pareto_df["dataset"].nunique()
    )
)

print("-" * 75)

print("Pareto objectives          :", PARETO_OBJECTIVES)

print("Objective direction        : MAXIMIZE")

print("-" * 75)

print(
    "Pareto evaluation saved    :",
    pareto_evaluation_path
)

print(
    "Pareto frontier saved      :",
    pareto_frontier_path
)

print(
    "Dataset summary saved      :",
    pareto_summary_path
)

print(
    "Validation report saved    :",
    pareto_validation_path
)

print("-" * 75)

print("Validation checks          :", len(pareto_validation_df))
print("Passed checks              :", passed_checks)
print("Failed checks              :", failed_checks)

print("Section 13 status          :", section_status)

print("=" * 75)

SECTION 13 — PARETO FRONTIER
Expected experiments       : 60
Complete input solutions   : 0
Evaluated solutions        : 0
Pareto-optimal solutions   : 0
Datasets evaluated         : 0
---------------------------------------------------------------------------
Pareto objectives          : ['privacy_score', 'fidelity_score', 'utility_score']
Objective direction        : MAXIMIZE
---------------------------------------------------------------------------
Pareto evaluation saved    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/pareto_evaluation.csv
Pareto frontier saved      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/pareto_frontier.csv
Dataset summary saved      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/tables/pareto_frontier_summary.csv
Validation report saved    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/metadata/section_13_validation.csv
-------------------

In [44]:
# ============================================================
# SECTION 14 — OPTIMAL OPERATING POINTS
# SPP-GAN PRIVACY-FIDELITY-UTILITY ANALYSIS
#
# Selection criteria:
#   1. Candidate must belong to the Pareto frontier.
#   2. All three trade-off scores must be available.
#   3. DP-configured variants must have verified accounting.
#   4. Achieved epsilon must not exceed the target epsilon.
#   5. Highest equal-weight trade-off score is selected.
#
# Missing evidence is never treated as successful DP evidence.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DP_TARGET_EPSILON = 5.0

OPTIMAL_COLUMNS = [
    "dataset",
    "variant",
    "repetition_id",
    "privacy_score",
    "fidelity_score",
    "utility_score",
    "tradeoff_score",
    "dp_configured",
    "dp_applied",
    "formal_dp_verified",
    "within_epsilon_target",
    "achieved_epsilon",
    "delta",
    "selection_method"
]

# ------------------------------------------------------------
# 2. VALIDATE INPUTS
# ------------------------------------------------------------

required_objects = [
    "aligned_df",
    "pareto_frontier_df",
    "KEY_COLUMNS",
    "DATASETS",
    "RESULTS_DIR"
]

for object_name in required_objects:

    if object_name not in globals():

        raise NameError(
            f"{object_name} is unavailable. "
            "Run the preceding Notebook 18 sections first."
        )

if not isinstance(pareto_frontier_df, pd.DataFrame):

    raise TypeError(
        "pareto_frontier_df must be a pandas DataFrame."
    )

if not isinstance(aligned_df, pd.DataFrame):

    raise TypeError(
        "aligned_df must be a pandas DataFrame."
    )

# ------------------------------------------------------------
# 3. BOOLEAN PARSING
# ------------------------------------------------------------

def parse_boolean_series(series):

    true_values = {
        "true", "1", "yes", "y", "verified", "pass"
    }

    false_values = {
        "false", "0", "no", "n", "unverified",
        "fail", "", "none", "null", "nan"
    }

    def convert(value):

        if pd.isna(value):
            return False

        if isinstance(value, (bool, np.bool_)):
            return bool(value)

        if isinstance(
            value,
            (int, float, np.integer, np.floating)
        ):
            return bool(value == 1)

        value = str(value).strip().lower()

        if value in true_values:
            return True

        if value in false_values:
            return False

        return False

    return series.apply(convert).astype(bool)


# ------------------------------------------------------------
# 4. PREPARE DP ACCOUNTING EVIDENCE
# ------------------------------------------------------------

DP_EVIDENCE_COLUMNS = [
    "dp_configured",
    "dp_applied",
    "achieved_epsilon",
    "delta",
    "privacy_scope",
    "accountant",
    "private_component",
    "valid_accounting_evidence"
]

dp_evidence_df = aligned_df[
    KEY_COLUMNS
].copy()

for col in DP_EVIDENCE_COLUMNS:

    if col in aligned_df.columns:

        dp_evidence_df[col] = aligned_df[col].values

    else:

        dp_evidence_df[col] = np.nan


# ------------------------------------------------------------
# 5. NORMALIZE DP FLAGS
# ------------------------------------------------------------

dp_evidence_df["dp_configured"] = (
    parse_boolean_series(
        dp_evidence_df["dp_configured"]
    )
)

dp_evidence_df["dp_applied"] = (
    parse_boolean_series(
        dp_evidence_df["dp_applied"]
    )
)

dp_evidence_df["valid_accounting_evidence"] = (
    parse_boolean_series(
        dp_evidence_df["valid_accounting_evidence"]
    )
)

# ------------------------------------------------------------
# 6. VALIDATE NUMERIC ACCOUNTING VALUES
# ------------------------------------------------------------

dp_evidence_df["achieved_epsilon"] = pd.to_numeric(
    dp_evidence_df["achieved_epsilon"],
    errors="coerce"
)

dp_evidence_df["delta"] = pd.to_numeric(
    dp_evidence_df["delta"],
    errors="coerce"
)

epsilon_valid = (
    dp_evidence_df["achieved_epsilon"].notna()
    & np.isfinite(
        dp_evidence_df["achieved_epsilon"]
    )
    & dp_evidence_df["achieved_epsilon"].gt(0)
)

delta_valid = (
    dp_evidence_df["delta"].notna()
    & np.isfinite(
        dp_evidence_df["delta"]
    )
    & dp_evidence_df["delta"].gt(0)
    & dp_evidence_df["delta"].lt(1)
)

# ------------------------------------------------------------
# 7. VALIDATE DP ACCOUNTING METADATA
# ------------------------------------------------------------

def nonempty_text(series):

    text_values = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return (
        text_values.ne("")
        & ~text_values.isin(
            ["nan", "none", "null"]
        )
    )


scope_valid = nonempty_text(
    dp_evidence_df["privacy_scope"]
)

accountant_valid = nonempty_text(
    dp_evidence_df["accountant"]
)

component_valid = nonempty_text(
    dp_evidence_df["private_component"]
)

# ------------------------------------------------------------
# 8. FORMAL DP VERIFICATION
# ------------------------------------------------------------
#
# A DP-configured run is not automatically a verified DP run.
#
# Formal evidence requires:
#   - DP configured
#   - DP actually applied
#   - valid accounting evidence
#   - positive finite achieved epsilon
#   - valid delta
#   - documented privacy scope
#   - documented accountant
#   - documented private component
#
# ------------------------------------------------------------

dp_evidence_df["formal_dp_verified"] = (

    dp_evidence_df["dp_configured"]

    & dp_evidence_df["dp_applied"]

    & dp_evidence_df["valid_accounting_evidence"]

    & epsilon_valid

    & delta_valid

    & scope_valid

    & accountant_valid

    & component_valid

)

# ------------------------------------------------------------
# 9. CHECK TARGET EPSILON
# ------------------------------------------------------------

dp_evidence_df["within_epsilon_target"] = (

    dp_evidence_df["formal_dp_verified"]

    & dp_evidence_df["achieved_epsilon"].le(
        DP_TARGET_EPSILON
    )

)

# ------------------------------------------------------------
# 10. PREPARE PARETO CANDIDATES
# ------------------------------------------------------------

PARETO_SCORE_COLUMNS = [
    "privacy_score",
    "fidelity_score",
    "utility_score",
    "tradeoff_score"
]

required_pareto_columns = (
    KEY_COLUMNS + PARETO_SCORE_COLUMNS
)

missing_pareto_columns = [
    col for col in required_pareto_columns
    if col not in pareto_frontier_df.columns
]

if missing_pareto_columns:

    raise ValueError(
        "Pareto frontier is missing required columns: "
        f"{missing_pareto_columns}"
    )

operating_candidates_df = pareto_frontier_df.copy()

# Ensure score fields are numeric.
for col in PARETO_SCORE_COLUMNS:

    operating_candidates_df[col] = pd.to_numeric(
        operating_candidates_df[col],
        errors="coerce"
    )

# Retain only complete and finite trade-off solutions.
complete_mask = (
    operating_candidates_df[PARETO_SCORE_COLUMNS]
    .notna()
    .all(axis=1)
)

finite_mask = np.isfinite(
    operating_candidates_df[PARETO_SCORE_COLUMNS]
    .to_numpy(dtype=float)
).all(axis=1)

operating_candidates_df = operating_candidates_df.loc[
    complete_mask & finite_mask
].copy()

# ------------------------------------------------------------
# 11. MERGE DP EVIDENCE
# ------------------------------------------------------------

dp_merge_columns = KEY_COLUMNS + [
    "dp_configured",
    "dp_applied",
    "formal_dp_verified",
    "within_epsilon_target",
    "achieved_epsilon",
    "delta",
    "privacy_scope",
    "accountant",
    "private_component"
]

if operating_candidates_df.empty:

    # Preserve a stable schema for empty outputs.
    for col in dp_merge_columns:

        if col not in operating_candidates_df.columns:

            if col in KEY_COLUMNS:
                continue

            operating_candidates_df[col] = pd.Series(
                dtype="object"
            )

else:

    operating_candidates_df = operating_candidates_df.merge(
        dp_evidence_df[dp_merge_columns],
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# 12. DEFINE ELIGIBILITY
# ------------------------------------------------------------

if "dp_configured" not in operating_candidates_df.columns:

    operating_candidates_df["dp_configured"] = False

if "dp_applied" not in operating_candidates_df.columns:

    operating_candidates_df["dp_applied"] = False

if "formal_dp_verified" not in operating_candidates_df.columns:

    operating_candidates_df["formal_dp_verified"] = False

if "within_epsilon_target" not in operating_candidates_df.columns:

    operating_candidates_df["within_epsilon_target"] = False


# Non-DP variants do not require DP accounting.
# DP-configured variants require verified accounting
# and achieved epsilon within the target.

operating_candidates_df["eligible"] = (

    ~parse_boolean_series(
        operating_candidates_df["dp_configured"]
    )

    |

    parse_boolean_series(
        operating_candidates_df["within_epsilon_target"]
    )

)

# Explicitly require complete trade-off scores.
operating_candidates_df["eligible"] &= (

    operating_candidates_df[
        PARETO_SCORE_COLUMNS
    ].notna().all(axis=1)

)

# ------------------------------------------------------------
# 13. SELECT OPTIMAL OPERATING POINTS
# ------------------------------------------------------------

optimal_records = []

for dataset in DATASETS:

    subset = operating_candidates_df.loc[

        (
            operating_candidates_df["dataset"]
            == dataset
        )

        & operating_candidates_df["eligible"]

    ].copy()

    if subset.empty:
        continue

    subset = subset.sort_values(

        by=[
            "tradeoff_score",
            "variant",
            "repetition_id"
        ],

        ascending=[
            False,
            True,
            True
        ]

    )

    best = subset.iloc[0]

    optimal_records.append({

        "dataset": dataset,

        "variant": best["variant"],

        "repetition_id": int(
            best["repetition_id"]
        ),

        "privacy_score": best["privacy_score"],

        "fidelity_score": best["fidelity_score"],

        "utility_score": best["utility_score"],

        "tradeoff_score": best["tradeoff_score"],

        "dp_configured": bool(
            best["dp_configured"]
        ),

        "dp_applied": bool(
            best["dp_applied"]
        ),

        "formal_dp_verified": bool(
            best["formal_dp_verified"]
        ),

        "within_epsilon_target": bool(
            best["within_epsilon_target"]
        ),

        "achieved_epsilon": best["achieved_epsilon"],

        "delta": best["delta"],

        "selection_method": (
            "pareto_frontier_equal_weight"
        )

    })


# ------------------------------------------------------------
# 14. CREATE OPTIMAL POINTS DATAFRAME
# ------------------------------------------------------------

optimal_df = pd.DataFrame(
    optimal_records
)

for col in OPTIMAL_COLUMNS:

    if col not in optimal_df.columns:

        optimal_df[col] = pd.Series(
            dtype="object"
        )

optimal_df = optimal_df[
    OPTIMAL_COLUMNS
]


# ------------------------------------------------------------
# 15. SAVE OUTPUTS
# ------------------------------------------------------------

optimal_output_path = (
    RESULTS_DIR / "optimal_operating_points.csv"
)

candidate_output_path = (
    RESULTS_DIR / "operating_point_candidates.csv"
)

operating_summary_path = (
    TABLES_DIR / "operating_point_summary.csv"
)

optimal_df.to_csv(
    optimal_output_path,
    index=False
)

operating_candidates_df.to_csv(
    candidate_output_path,
    index=False
)

# Dataset-level selection coverage.
operating_summary_records = []

for dataset in DATASETS:

    dataset_candidates = operating_candidates_df.loc[
        operating_candidates_df["dataset"] == dataset
    ]

    dataset_eligible = dataset_candidates.loc[
        dataset_candidates["eligible"]
    ]

    dataset_selected = optimal_df.loc[
        optimal_df["dataset"] == dataset
    ]

    operating_summary_records.append({

        "dataset": dataset,

        "pareto_candidates": len(dataset_candidates),

        "eligible_candidates": len(dataset_eligible),

        "selected_operating_points": len(dataset_selected),

        "status": (
            "SELECTED"
            if len(dataset_selected) > 0
            else "NO_ELIGIBLE_OPERATING_POINT"
        )

    })

operating_summary_df = pd.DataFrame(
    operating_summary_records
)

operating_summary_df.to_csv(
    operating_summary_path,
    index=False
)


# ------------------------------------------------------------
# 16. VALIDATION
# ------------------------------------------------------------

validation_checks = [

    {
        "check": "optimal_output_schema_valid",
        "status": list(optimal_df.columns) == OPTIMAL_COLUMNS
    },

    {
        "check": "selected_records_are_eligible",
        "status": (
            optimal_df.empty
            or optimal_df["dataset"].isin(DATASETS).all()
        )
    },

    {
        "check": "no_duplicate_dataset_selections",
        "status": (
            not optimal_df["dataset"].duplicated().any()
        )
    },

    {
        "check": "dp_candidates_have_verified_accounting",
        "status": (
            operating_candidates_df.loc[
                operating_candidates_df["dp_configured"].fillna(False),
                "eligible"
            ].eq(False).all()
            or operating_candidates_df.loc[
                operating_candidates_df["eligible"]
                & operating_candidates_df["dp_configured"].fillna(False),
                "formal_dp_verified"
            ].eq(True).all()
        )
    },

    {
        "check": "output_files_saved",
        "status": (
            optimal_output_path.exists()
            and candidate_output_path.exists()
            and operating_summary_path.exists()
        )
    }

]

operating_validation_df = pd.DataFrame(
    validation_checks
)

operating_validation_path = (
    METADATA_DIR / "section_14_validation.csv"
)

operating_validation_df.to_csv(
    operating_validation_path,
    index=False
)

passed_checks = int(
    operating_validation_df["status"].sum()
)

failed_checks = (
    len(operating_validation_df) - passed_checks
)

section_status = (
    "PASS"
    if failed_checks == 0
    else "FAILED"
)


# ------------------------------------------------------------
# 17. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 14 — OPTIMAL OPERATING POINTS")
print("=" * 75)

print("Target epsilon              :", DP_TARGET_EPSILON)

print(
    "Pareto candidates           :",
    len(operating_candidates_df)
)

print(
    "Eligible points             :",
    int(operating_candidates_df["eligible"].sum())
)

print(
    "Selected points             :",
    len(optimal_df)
)

print("-" * 75)

print(
    "Formal DP verified records  :",
    int(dp_evidence_df["formal_dp_verified"].sum())
)

print(
    "Within epsilon target       :",
    int(dp_evidence_df["within_epsilon_target"].sum())
)

print("-" * 75)

print("Optimal points saved        :", optimal_output_path)

print("Candidate records saved     :", candidate_output_path)

print("Dataset summary saved       :", operating_summary_path)

print("Validation report saved     :", operating_validation_path)

print("-" * 75)

print("Validation checks           :", len(operating_validation_df))
print("Passed checks               :", passed_checks)
print("Failed checks               :", failed_checks)

print("Section 14 status           :", section_status)

print("=" * 75)

SECTION 14 — OPTIMAL OPERATING POINTS
Target epsilon              : 5.0
Pareto candidates           : 0
Eligible points             : 0
Selected points             : 0
---------------------------------------------------------------------------
Formal DP verified records  : 0
Within epsilon target       : 0
---------------------------------------------------------------------------
Optimal points saved        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/optimal_operating_points.csv
Candidate records saved     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/results/operating_point_candidates.csv
Dataset summary saved       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/tables/operating_point_summary.csv
Validation report saved     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/metadata/section_14_validation.csv
----------------------------------------------------------------------

In [45]:
# ============================================================
# SECTION 15 — DATASET-LEVEL TRADE-OFFS
# ============================================================

dataset_records = []

for dataset in DATASETS:

    subset = three_way_df.loc[
        three_way_df["dataset"] == dataset
    ]

    record = {
        "dataset": dataset,
        "expected_runs": len(VARIANTS) * len(REPETITIONS),
        "valid_runs": len(subset),
        "variants_evaluated": subset["variant"].nunique()
    }

    for metric in SCORE_COLUMNS + ["tradeoff_score"]:
        record[f"{metric}_mean"] = (
            subset[metric].mean()
            if len(subset) else np.nan
        )

        record[f"{metric}_std"] = (
            subset[metric].std()
            if len(subset) > 1 else np.nan
        )

    dataset_records.append(record)

dataset_tradeoff_df = pd.DataFrame(dataset_records)

dataset_tradeoff_df.to_csv(
    TABLES_DIR / "dataset_level_tradeoffs.csv",
    index=False
)

print(dataset_tradeoff_df.to_string(index=False))

       dataset  expected_runs  valid_runs  variants_evaluated  privacy_score_mean  privacy_score_std  fidelity_score_mean  fidelity_score_std  utility_score_mean  utility_score_std  tradeoff_score_mean  tradeoff_score_std
  adult_income             20           0                   0                 NaN                NaN                  NaN                 NaN                 NaN                NaN                  NaN                 NaN
bank_marketing             20           0                   0                 NaN                NaN                  NaN                 NaN                 NaN                NaN                  NaN                 NaN
diabetes_130us             20           0                   0                 NaN                NaN                  NaN                 NaN                 NaN                NaN                  NaN                 NaN


In [46]:
# ============================================================
# SECTION 16 — MODEL-LEVEL TRADE-OFFS
# ============================================================

model_records = []

for variant in VARIANTS:

    subset = three_way_df.loc[
        three_way_df["variant"] == variant
    ]

    record = {
        "variant": variant,
        "dp_configured": variant in DP_VARIANTS,
        "expected_runs": len(DATASETS) * len(REPETITIONS),
        "valid_runs": len(subset),
        "datasets_evaluated": subset["dataset"].nunique()
    }

    for metric in SCORE_COLUMNS + ["tradeoff_score"]:
        record[f"{metric}_mean"] = (
            subset[metric].mean()
            if len(subset) else np.nan
        )

        record[f"{metric}_std"] = (
            subset[metric].std()
            if len(subset) > 1 else np.nan
        )

    model_records.append(record)

model_tradeoff_df = pd.DataFrame(model_records)

model_tradeoff_df.to_csv(
    TABLES_DIR / "model_level_tradeoffs.csv",
    index=False
)

print(model_tradeoff_df.to_string(index=False))

                 variant  dp_configured  expected_runs  valid_runs  datasets_evaluated  privacy_score_mean  privacy_score_std  fidelity_score_mean  fidelity_score_std  utility_score_mean  utility_score_std  tradeoff_score_mean  tradeoff_score_std
                gan_only          False             15           0                   0                 NaN                NaN                  NaN                 NaN                 NaN                NaN                  NaN                 NaN
statistical_guidance_gan          False             15           0                   0                 NaN                NaN                  NaN                 NaN                 NaN                NaN                  NaN                 NaN
                  gan_dp           True             15           0                   0                 NaN                NaN                  NaN                 NaN                 NaN                NaN                  NaN                 NaN
            

In [47]:
# ============================================================
# SECTION 17 — SAVE TRADE-OFF RESULTS
# ============================================================

SOURCE_PROVENANCE = {
    "fidelity": fidelity_source,
    "utility": utility_source,
    "privacy": privacy_source,
    "dp_accounting": dp_source
}

COVERAGE = {
    "expected_experiments": EXPECTED_RECORDS,
    "fidelity_source_rows": len(fidelity_raw),
    "utility_source_rows": len(utility_raw),
    "privacy_source_rows": len(privacy_raw),
    "dp_source_rows": len(dp_raw),
    "aligned_records": len(aligned_df),
    "privacy_fidelity_pairs": len(privacy_fidelity_df),
    "privacy_utility_pairs": len(privacy_utility_df),
    "fidelity_utility_pairs": len(fidelity_utility_df),
    "complete_three_way_records": len(three_way_df),
    "pareto_optimal_records": len(pareto_frontier_df),
    "selected_operating_points": len(optimal_df),
    "formal_dp_evidence_records": int(
        dp_evidence_df["formal_dp_verified"].sum()
    )
}

coverage_df = pd.DataFrame([
    {"measure": key, "value": value}
    for key, value in COVERAGE.items()
])

coverage_df.to_csv(
    VALIDATION_DIR / "tradeoff_coverage.csv",
    index=False
)

configuration = {
    "notebook": NOTEBOOK_ID,
    "name": NOTEBOOK_NAME,
    "execution_utc": RUN_TIMESTAMP,
    "project_root": str(PROJECT_ROOT),
    "master_seed": MASTER_SEED,
    "datasets": DATASETS,
    "variants": VARIANTS,
    "repetitions": REPETITIONS,
    "expected_records": EXPECTED_RECORDS,
    "dp_target_epsilon": DP_TARGET_EPSILON,
    "dp_delta_rule": DP_DELTA_RULE,
    "selected_metrics": SELECTED_METRICS,
    "tradeoff_weights": TRADEOFF_WEIGHTS,
    "source_provenance": SOURCE_PROVENANCE,
    "coverage": COVERAGE
}

configuration_path = (
    METADATA_DIR / "notebook_18_configuration.json"
)

with open(configuration_path, "w") as file:
    json.dump(configuration, file, indent=2)

required_outputs = [
    RESULTS_DIR / "aligned_experimental_conditions.csv",
    RESULTS_DIR / "normalized_metrics.csv",
    RESULTS_DIR / "three_way_tradeoff_results.csv",
    RESULTS_DIR / "pareto_frontier.csv",
    RESULTS_DIR / "optimal_operating_points.csv",
    TABLES_DIR / "dataset_level_tradeoffs.csv",
    TABLES_DIR / "model_level_tradeoffs.csv",
    configuration_path
]

save_validation = pd.DataFrame([
    {
        "artifact": str(path.relative_to(NOTEBOOK_ROOT)),
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else 0
    }
    for path in required_outputs
])

save_validation.to_csv(
    VALIDATION_DIR / "artifact_save_validation.csv",
    index=False
)

SAVE_VALID = bool(save_validation["exists"].all())

print("Required artifacts :", len(required_outputs))
print("Artifacts saved    :", int(save_validation["exists"].sum()))
print("Save validation    :", "PASS" if SAVE_VALID else "FAIL")

Required artifacts : 8
Artifacts saved    : 8
Save validation    : PASS


In [49]:
# ============================================================
# SECTION 18 — GENERATE TRADE-OFF FIGURES
# SPP-GAN PRIVACY-FIDELITY-UTILITY ANALYSIS
#
# Generates:
#   1. Privacy vs Fidelity scatter plot
#   2. Privacy vs Utility scatter plot
#   3. Fidelity vs Utility scatter plot
#   4. Three-way Pareto frontier
#   5. Dataset-level trade-off comparison
#   6. Model-level trade-off comparison
#   7. Experimental evidence coverage
#   8. Metric availability by dataset
#
# Empty trade-off results are not fabricated.
# Coverage figures explicitly identify missing evidence.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

FIGURE_DPI = 180

VARIANT_LABELS = {
    "gan_only": "GAN Only",
    "statistical_guidance_gan": "Statistical + GAN",
    "gan_dp": "GAN + DP",
    "full_sppgan": "Full SPP-GAN"
}

FIGURE_MANIFEST = []
FIGURE_STATUS = []

# ------------------------------------------------------------
# 2. VALIDATE REQUIRED OBJECTS
# ------------------------------------------------------------

required_objects = [
    "privacy_fidelity_df",
    "privacy_utility_df",
    "fidelity_utility_df",
    "pareto_df",
    "three_way_df",
    "three_way_summary_df",
    "dataset_tradeoff_df",
    "model_tradeoff_df",
    "aligned_df",
    "tradeoff_df",
    "DATASETS",
    "VARIANTS",
    "FIGURES_DIR",
    "MANIFESTS_DIR"
]

for object_name in required_objects:

    if object_name not in globals():

        raise NameError(
            f"{object_name} is unavailable. "
            "Run the preceding Notebook 18 sections first."
        )

# ------------------------------------------------------------
# 3. FIGURE-SAVING HELPER
# ------------------------------------------------------------

def register_figure(path, figure_type, status):

    FIGURE_MANIFEST.append({

        "figure_path": str(path),

        "figure_type": figure_type,

        "status": status,

        "exists": path.is_file(),

        "size_bytes": (
            path.stat().st_size
            if path.is_file()
            else 0
        )

    })


# ------------------------------------------------------------
# 4. PAIRWISE SCATTER PLOTS
# ------------------------------------------------------------

def save_scatter(
    frame,
    x_label,
    y_label,
    filename,
    title,
    x_column="x_score",
    y_column="y_score"
):

    path = FIGURES_DIR / filename

    if frame.empty:

        FIGURE_STATUS.append({

            "figure": filename,

            "status": "NOT_GENERATED_NO_VALID_RECORDS",

            "valid_records": 0

        })

        return False

    required_columns = [
        "variant",
        x_column,
        y_column
    ]

    missing = [
        col for col in required_columns
        if col not in frame.columns
    ]

    if missing:

        raise ValueError(
            f"Missing scatter plot columns: {missing}"
        )

    plot_df = frame.copy()

    plot_df[x_column] = pd.to_numeric(
        plot_df[x_column],
        errors="coerce"
    )

    plot_df[y_column] = pd.to_numeric(
        plot_df[y_column],
        errors="coerce"
    )

    plot_df = plot_df.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna(
        subset=[x_column, y_column]
    )

    if plot_df.empty:

        FIGURE_STATUS.append({

            "figure": filename,

            "status": "NOT_GENERATED_NO_VALID_RECORDS",

            "valid_records": 0

        })

        return False

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    for variant in VARIANTS:

        subset = plot_df.loc[
            plot_df["variant"] == variant
        ]

        if subset.empty:
            continue

        ax.scatter(

            subset[x_column],

            subset[y_column],

            label=VARIANT_LABELS.get(
                variant,
                variant
            ),

            s=55,

            alpha=0.75

        )

    ax.set_xlabel(x_label)

    ax.set_ylabel(y_label)

    ax.set_title(title)

    ax.grid(alpha=0.25)

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.close(fig)

    register_figure(
        path,
        "pairwise_tradeoff",
        "GENERATED"
    )

    FIGURE_STATUS.append({

        "figure": filename,

        "status": "GENERATED",

        "valid_records": len(plot_df)

    })

    return True


# ------------------------------------------------------------
# 5. PRIVACY VS FIDELITY
# ------------------------------------------------------------

save_scatter(

    privacy_fidelity_df,

    "Formal DP Privacy Protection (Higher = Better)",

    "Fidelity Score (Higher = Better)",

    "privacy_vs_fidelity.png",

    "Privacy–Fidelity Trade-off"

)


# ------------------------------------------------------------
# 6. PRIVACY VS UTILITY
# ------------------------------------------------------------

save_scatter(

    privacy_utility_df,

    "Formal DP Privacy Protection (Higher = Better)",

    "Utility Score (Higher = Better)",

    "privacy_vs_utility.png",

    "Privacy–Utility Trade-off"

)


# ------------------------------------------------------------
# 7. FIDELITY VS UTILITY
# ------------------------------------------------------------

save_scatter(

    fidelity_utility_df,

    "Fidelity Score (Higher = Better)",

    "Utility Score (Higher = Better)",

    "fidelity_vs_utility.png",

    "Fidelity–Utility Trade-off"

)


# ------------------------------------------------------------
# 8. THREE-WAY PARETO FRONTIER
# ------------------------------------------------------------

pareto_path = (
    FIGURES_DIR / "three_way_pareto_frontier.png"
)

pareto_required = [
    "privacy_score",
    "fidelity_score",
    "utility_score",
    "pareto_optimal"
]

missing_pareto_columns = [
    col for col in pareto_required
    if col not in pareto_df.columns
]

if missing_pareto_columns:

    raise ValueError(
        "Missing Pareto columns: "
        f"{missing_pareto_columns}"
    )

pareto_plot_df = pareto_df.copy()

for col in pareto_required[:3]:

    pareto_plot_df[col] = pd.to_numeric(
        pareto_plot_df[col],
        errors="coerce"
    )

pareto_plot_df = pareto_plot_df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=pareto_required[:3]
)

if not pareto_plot_df.empty:

    fig = plt.figure(
        figsize=(9, 7)
    )

    ax = fig.add_subplot(
        111,
        projection="3d"
    )

    for is_pareto, label in [

        (False, "Dominated"),

        (True, "Pareto-optimal")

    ]:

        subset = pareto_plot_df.loc[
            pareto_plot_df["pareto_optimal"] == is_pareto
        ]

        if subset.empty:
            continue

        ax.scatter(

            subset["privacy_score"],

            subset["fidelity_score"],

            subset["utility_score"],

            label=label,

            s=75 if is_pareto else 30,

            alpha=0.9 if is_pareto else 0.3

        )

    ax.set_xlabel("Formal DP Privacy Protection")

    ax.set_ylabel("Fidelity")

    ax.set_zlabel("Utility")

    ax.set_title(
        "Three-Way Pareto Trade-off"
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        pareto_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.close(fig)

    register_figure(
        pareto_path,
        "three_way_pareto",
        "GENERATED"
    )

else:

    FIGURE_STATUS.append({

        "figure": "three_way_pareto_frontier.png",

        "status": "NOT_GENERATED_NO_VALID_RECORDS",

        "valid_records": 0

    })


# ------------------------------------------------------------
# 9. DATASET-LEVEL TRADE-OFF COMPARISON
# ------------------------------------------------------------

dataset_figure_path = (
    FIGURES_DIR / "dataset_level_tradeoffs.png"
)

if (
    not dataset_tradeoff_df.empty
    and "tradeoff_score_mean" in dataset_tradeoff_df.columns
):

    plot_df = dataset_tradeoff_df.copy()

    plot_df["tradeoff_score_mean"] = pd.to_numeric(
        plot_df["tradeoff_score_mean"],
        errors="coerce"
    )

    plot_df = plot_df.dropna(
        subset=["tradeoff_score_mean"]
    )

else:

    plot_df = pd.DataFrame()

if not plot_df.empty:

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.bar(
        plot_df["dataset"],
        plot_df["tradeoff_score_mean"]
    )

    ax.set_title(
        "Dataset-Level Composite Trade-off"
    )

    ax.set_ylabel(
        "Mean Composite Trade-off Score"
    )

    ax.set_ylim(0, 1)

    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    fig.savefig(
        dataset_figure_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.close(fig)

    register_figure(
        dataset_figure_path,
        "dataset_level_tradeoff",
        "GENERATED"
    )

else:

    FIGURE_STATUS.append({

        "figure": "dataset_level_tradeoffs.png",

        "status": "NOT_GENERATED_NO_VALID_RECORDS",

        "valid_records": 0

    })


# ------------------------------------------------------------
# 10. MODEL-LEVEL TRADE-OFF COMPARISON
# ------------------------------------------------------------

model_figure_path = (
    FIGURES_DIR / "model_level_tradeoffs.png"
)

if (
    not model_tradeoff_df.empty
    and "tradeoff_score_mean" in model_tradeoff_df.columns
):

    plot_df = model_tradeoff_df.copy()

    plot_df["tradeoff_score_mean"] = pd.to_numeric(
        plot_df["tradeoff_score_mean"],
        errors="coerce"
    )

    plot_df = plot_df.dropna(
        subset=["tradeoff_score_mean"]
    )

else:

    plot_df = pd.DataFrame()

if not plot_df.empty:

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    ax.bar(

        [
            VARIANT_LABELS.get(v, v)
            for v in plot_df["variant"]
        ],

        plot_df["tradeoff_score_mean"]

    )

    ax.set_title(
        "Model-Level Composite Trade-off"
    )

    ax.set_ylabel(
        "Mean Composite Trade-off Score"
    )

    ax.set_ylim(0, 1)

    ax.tick_params(
        axis="x",
        rotation=20
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    fig.savefig(
        model_figure_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.close(fig)

    register_figure(
        model_figure_path,
        "model_level_tradeoff",
        "GENERATED"
    )

else:

    FIGURE_STATUS.append({

        "figure": "model_level_tradeoffs.png",

        "status": "NOT_GENERATED_NO_VALID_RECORDS",

        "valid_records": 0

    })


# ------------------------------------------------------------
# 11. EXPERIMENTAL EVIDENCE COVERAGE FIGURE
# ------------------------------------------------------------
# This is a diagnostic figure, not a trade-off result.
# It reports the evidence actually available.
# ------------------------------------------------------------

coverage_path = (
    FIGURES_DIR / "experimental_evidence_coverage.png"
)

coverage_labels = [
    "Fidelity",
    "Utility",
    "Empirical Privacy",
    "Formal DP"
]

coverage_values = [

    int(
        aligned_df.filter(
            regex="^(wasserstein_distance|js_divergence|mmd|ks_similarity|fidelity_score)$"
        ).notna().any(axis=1).sum()
    ),

    int(
        aligned_df.filter(
            regex="^(accuracy|macro_f1|weighted_f1|tstr_f1|downstream_utility)$"
        ).notna().any(axis=1).sum()
    ),

    int(
        aligned_df.filter(
            regex="^(membership_inference_auc|mia_auc|reidentification_risk|disclosure_risk)$"
        ).notna().any(axis=1).sum()
    ),

    int(
        tradeoff_df["formal_dp_verified"].sum()
    ) if "formal_dp_verified" in tradeoff_df.columns else 0

]

fig, ax = plt.subplots(
    figsize=(9, 5)
)

bars = ax.bar(
    coverage_labels,
    coverage_values
)

ax.set_title(
    "Experimental Evidence Coverage"
)

ax.set_ylabel(
    "Records with Available Evidence"
)

ax.set_ylim(
    0,
    max(5, len(aligned_df) * 1.1)
)

ax.set_xticks(
    range(len(coverage_labels))
)

ax.set_xticklabels(
    coverage_labels,
    rotation=15
)

ax.grid(
    axis="y",
    alpha=0.25
)

for bar, value in zip(bars, coverage_values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(value),
        ha="center",
        va="bottom"
    )

fig.tight_layout()

fig.savefig(
    coverage_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.close(fig)

register_figure(
    coverage_path,
    "evidence_coverage",
    "GENERATED"
)


# ------------------------------------------------------------
# 12. DATASET-LEVEL EVIDENCE AVAILABILITY
# ------------------------------------------------------------

dataset_coverage_path = (
    FIGURES_DIR / "dataset_evidence_coverage.png"
)

dataset_coverage_records = []

for dataset in DATASETS:

    subset = aligned_df.loc[
        aligned_df["dataset"] == dataset
    ]

    dataset_coverage_records.append({

        "dataset": dataset,

        "expected_records": len(subset),

        "fidelity_records": int(
            subset.filter(
                regex="^(wasserstein_distance|js_divergence|mmd|ks_similarity|fidelity_score)$"
            ).notna().any(axis=1).sum()
        ),

        "utility_records": int(
            subset.filter(
                regex="^(accuracy|macro_f1|weighted_f1|tstr_f1|downstream_utility)$"
            ).notna().any(axis=1).sum()
        ),

        "empirical_privacy_records": int(
            subset.filter(
                regex="^(membership_inference_auc|mia_auc|reidentification_risk|disclosure_risk)$"
            ).notna().any(axis=1).sum()
        )

    })

dataset_coverage_df = pd.DataFrame(
    dataset_coverage_records
)

if not dataset_coverage_df.empty:

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    x = np.arange(
        len(dataset_coverage_df)
    )

    width = 0.22

    for index, metric in enumerate([

        "fidelity_records",

        "utility_records",

        "empirical_privacy_records"

    ]):

        ax.bar(

            x + (index - 1) * width,

            dataset_coverage_df[metric],

            width,

            label=metric.replace(
                "_records",
                ""
            ).replace(
                "_",
                " "
            ).title()

        )

    ax.set_xticks(x)

    ax.set_xticklabels(
        dataset_coverage_df["dataset"],
        rotation=15
    )

    ax.set_ylabel(
        "Records with Available Evidence"
    )

    ax.set_title(
        "Dataset-Level Evidence Availability"
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    fig.savefig(
        dataset_coverage_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.close(fig)

    register_figure(
        dataset_coverage_path,
        "dataset_evidence_coverage",
        "GENERATED"
    )


# ------------------------------------------------------------
# 13. SAVE FIGURE MANIFEST
# ------------------------------------------------------------

figure_manifest_df = pd.DataFrame(
    FIGURE_MANIFEST,
    columns=[
        "figure_path",
        "figure_type",
        "status",
        "exists",
        "size_bytes"
    ]
)

figure_manifest_path = (
    MANIFESTS_DIR / "figure_manifest.csv"
)

figure_manifest_df.to_csv(
    figure_manifest_path,
    index=False
)


# ------------------------------------------------------------
# 14. SAVE FIGURE STATUS REPORT
# ------------------------------------------------------------

figure_status_df = pd.DataFrame(
    FIGURE_STATUS,
    columns=[
        "figure",
        "status",
        "valid_records"
    ]
)

figure_status_path = (
    METADATA_DIR / "section_18_figure_status.csv"
)

figure_status_df.to_csv(
    figure_status_path,
    index=False
)


# ------------------------------------------------------------
# 15. VALIDATE FIGURE OUTPUTS
# ------------------------------------------------------------

generated_count = int(
    figure_manifest_df["exists"].sum()
)

failed_count = int(
    (
        figure_manifest_df["exists"] == False
    ).sum()
)

validation_checks = [

    {
        "check": "figure_manifest_created",
        "status": figure_manifest_path.is_file()
    },

    {
        "check": "figure_status_report_created",
        "status": figure_status_path.is_file()
    },

    {
        "check": "all_registered_figures_exist",
        "status": failed_count == 0
    },

    {
        "check": "evidence_coverage_figure_created",
        "status": coverage_path.is_file()
    },

    {
        "check": "dataset_coverage_figure_created",
        "status": dataset_coverage_path.is_file()
    }

]

figure_validation_df = pd.DataFrame(
    validation_checks
)

figure_validation_path = (
    METADATA_DIR / "section_18_validation.csv"
)

figure_validation_df.to_csv(
    figure_validation_path,
    index=False
)

passed_checks = int(
    figure_validation_df["status"].sum()
)

failed_checks = (
    len(figure_validation_df) - passed_checks
)

section_status = (
    "PASS"
    if failed_checks == 0
    else "FAILED"
)


# ------------------------------------------------------------
# 16. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("SECTION 18 — GENERATE TRADE-OFF FIGURES")
print("=" * 75)

print("Trade-off figures generated :", generated_count)

print(
    "Trade-off figures skipped   :",
    int(
        (
            figure_status_df["status"]
            == "NOT_GENERATED_NO_VALID_RECORDS"
        ).sum()
    )
)

print("Total figure manifest rows  :", len(figure_manifest_df))

print("-" * 75)

print(
    "Evidence coverage figure   :",
    coverage_path.is_file()
)

print(
    "Dataset coverage figure    :",
    dataset_coverage_path.is_file()
)

print("Figure directory           :", FIGURES_DIR)

print("Figure manifest saved      :", figure_manifest_path)

print("Figure status saved        :", figure_status_path)

print("-" * 75)

print("Validation checks          :", len(figure_validation_df))

print("Passed checks              :", passed_checks)

print("Failed checks              :", failed_checks)

print("Section 18 status          :", section_status)

print("=" * 75)

SECTION 18 — GENERATE TRADE-OFF FIGURES
Trade-off figures generated : 2
Trade-off figures skipped   : 6
Total figure manifest rows  : 2
---------------------------------------------------------------------------
Evidence coverage figure   : True
Dataset coverage figure    : True
Figure directory           : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/figures
Figure manifest saved      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/manifests/figure_manifest.csv
Figure status saved        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_18/metadata/section_18_figure_status.csv
---------------------------------------------------------------------------
Validation checks          : 5
Passed checks              : 5
Failed checks              : 0
Section 18 status          : PASS


In [52]:

# ============================================================
# SECTION 19 — COMPLETION SUMMARY
# SPP-GAN PRIVACY-FIDELITY-UTILITY TRADE-OFF ANALYSIS
#
# Separates:
#   1. Notebook execution completion
#   2. Research evaluation completeness
#   3. Evidence availability
#   4. Artifact integrity
#
# Missing empirical evidence is never fabricated.
# ============================================================

import os
import json
import hashlib
import gc

import numpy as np
import pandas as pd

from datetime import datetime, timezone


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPECTED_DP_RECORDS = (
    len(DATASETS)
    * len(REPETITIONS)
    * len(DP_VARIANTS)
)

EXPECTED_NON_DP_RECORDS = (
    len(DATASETS)
    * len(REPETITIONS)
    * (len(VARIANTS) - len(DP_VARIANTS))
)

EXPECTED_RECORDS = (
    len(DATASETS)
    * len(REPETITIONS)
    * len(VARIANTS)
)

assert EXPECTED_RECORDS == 60, (
    f"Unexpected experimental design: {EXPECTED_RECORDS}"
)

assert EXPECTED_DP_RECORDS == 30, (
    f"Unexpected DP experiment count: {EXPECTED_DP_RECORDS}"
)


# ============================================================
# 2. SAFE VALIDATION HELPERS
# ============================================================

def count_numeric_evidence(df, column):

    if (
        not isinstance(df, pd.DataFrame)
        or column not in df.columns
    ):
        return 0

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    return int(
        values.replace(
            [np.inf, -np.inf],
            np.nan
        ).notna().sum()
    )


def count_true_values(df, column):

    if (
        not isinstance(df, pd.DataFrame)
        or column not in df.columns
    ):
        return 0

    values = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return int(
        values.isin([
            "true",
            "1",
            "yes"
        ]).sum()
    )


def dataframe_exists(name):

    return (
        name in globals()
        and isinstance(globals()[name], pd.DataFrame)
    )


def file_exists(path):

    return (
        path is not None
        and os.path.isfile(str(path))
    )


# ============================================================
# 3. EXPERIMENTAL GRID VALIDATION
# ============================================================

aligned_records = (
    len(aligned_df)
    if dataframe_exists("aligned_df")
    else 0
)

required_key_columns = [
    col for col in KEY_COLUMNS
    if col in aligned_df.columns
] if dataframe_exists("aligned_df") else []

key_schema_valid = (
    len(required_key_columns) == len(KEY_COLUMNS)
)

unique_experimental_conditions = (

    key_schema_valid

    and not aligned_df.duplicated(
        KEY_COLUMNS
    ).any()

)

experimental_grid_valid = (

    aligned_records == EXPECTED_RECORDS

    and unique_experimental_conditions

)


# ============================================================
# 4. EMPIRICAL FIDELITY AND UTILITY EVIDENCE
# ============================================================

fidelity_records = (
    count_numeric_evidence(
        tradeoff_df,
        "fidelity_score"
    )
    if dataframe_exists("tradeoff_df")
    else 0
)

utility_records = (
    count_numeric_evidence(
        tradeoff_df,
        "utility_score"
    )
    if dataframe_exists("tradeoff_df")
    else 0
)


# ============================================================
# 5. EMPIRICAL PRIVACY EVIDENCE
# ============================================================
# Empirical attack metrics are separate from formal DP.
# A formal privacy protection score is NOT an attack metric.

EMPIRICAL_PRIVACY_COLUMNS = [

    "membership_inference_auc",

    "mia_auc",

    "reidentification_risk",

    "disclosure_risk",

    "nearest_neighbor_disclosure_risk",

    "record_linkage_risk"

]

available_empirical_privacy_columns = [

    col for col in EMPIRICAL_PRIVACY_COLUMNS

    if (
        dataframe_exists("tradeoff_df")
        and col in tradeoff_df.columns
    )

]

if available_empirical_privacy_columns:

    empirical_privacy_records = int(

        tradeoff_df[
            available_empirical_privacy_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .notna()
        .any(axis=1)
        .sum()

    )

else:

    empirical_privacy_records = 0


# ============================================================
# 6. FORMAL DP ACCOUNTING EVIDENCE
# ============================================================

formal_dp_records = (
    count_true_values(
        dp_evidence_df,
        "formal_dp_verified"
    )
    if dataframe_exists("dp_evidence_df")
    else 0
)

configured_dp_records = (
    count_true_values(
        dp_evidence_df,
        "dp_configured"
    )
    if dataframe_exists("dp_evidence_df")
    else 0
)

dp_applied_records = (
    count_true_values(
        dp_evidence_df,
        "dp_applied"
    )
    if dataframe_exists("dp_evidence_df")
    else 0
)

formal_dp_complete = (

    configured_dp_records == EXPECTED_DP_RECORDS

    and dp_applied_records == EXPECTED_DP_RECORDS

    and formal_dp_records == EXPECTED_DP_RECORDS

)


# ============================================================
# 7. THREE-WAY TRADE-OFF AND PARETO RESULTS
# ============================================================

complete_three_way_records = (
    len(three_way_df)
    if dataframe_exists("three_way_df")
    else 0
)

pareto_records = (
    len(pareto_frontier_df)
    if dataframe_exists("pareto_frontier_df")
    else 0
)

optimal_records = (
    len(optimal_df)
    if dataframe_exists("optimal_df")
    else 0
)


# ============================================================
# 8. FIGURE MANIFEST VALIDATION
# ============================================================

figure_manifest_path = (
    MANIFESTS_DIR / "figure_manifest.csv"
)

if file_exists(figure_manifest_path):

    figure_manifest_df = pd.read_csv(
        figure_manifest_path
    )

    if "exists" in figure_manifest_df.columns:

        generated_figure_count = count_true_values(
            figure_manifest_df,
            "exists"
        )

    else:

        generated_figure_count = len(
            figure_manifest_df
        )

else:

    figure_manifest_df = pd.DataFrame()

    generated_figure_count = 0


figure_manifest_valid = (
    file_exists(figure_manifest_path)
)


# ============================================================
# 9. CORE ARTIFACT VALIDATION
# ============================================================
# Use Section 17's SAVE_VALID result rather than imposing
# a second, potentially inconsistent list of filenames.

section_17_artifacts_valid = bool(
    globals().get("SAVE_VALID", False)
)

core_directories_valid = all([

    RESULTS_DIR.exists(),

    TABLES_DIR.exists(),

    FIGURES_DIR.exists(),

    MANIFESTS_DIR.exists(),

    METADATA_DIR.exists(),

    VALIDATION_DIR.exists()

])

core_outputs_valid = (

    section_17_artifacts_valid

    and core_directories_valid

    and figure_manifest_valid

)


# ============================================================
# 10. EXECUTION COMPLETION
# ============================================================
# Missing research evidence does not mean the notebook
# failed to execute.
#
# Execution is based on:
#   - Valid experimental grid
#   - Successful Section 17 artifact saving
#   - Valid core output directories
#   - Figure manifest creation

execution_complete = bool(

    experimental_grid_valid

    and core_outputs_valid

)


# ============================================================
# 11. RESEARCH EVALUATION COMPLETENESS
# ============================================================
# Full evaluation requires:
#   - Fidelity measurements
#   - Utility measurements
#   - Empirical privacy measurements
#   - Formal DP evidence
#   - Complete three-way trade-off records

research_evaluation_complete = bool(

    fidelity_records == EXPECTED_RECORDS

    and utility_records == EXPECTED_RECORDS

    and empirical_privacy_records == EXPECTED_RECORDS

    and formal_dp_complete

    and complete_three_way_records == EXPECTED_RECORDS

)


# ============================================================
# 12. FINAL NOTEBOOK STATUS
# ============================================================

if not execution_complete:

    FINAL_STATUS = "INCOMPLETE"

elif not research_evaluation_complete:

    FINAL_STATUS = "COMPLETED_WITH_MISSING_EVIDENCE"

else:

    FINAL_STATUS = "COMPLETED"


# ============================================================
# 13. DETAILED VALIDATION REPORT
# ============================================================

VALIDATION_CHECKS = [

    {
        "check": "experimental_grid",
        "observed": aligned_records,
        "expected": EXPECTED_RECORDS,
        "passed": experimental_grid_valid
    },

    {
        "check": "fidelity_evidence",
        "observed": fidelity_records,
        "expected": EXPECTED_RECORDS,
        "passed": (
            fidelity_records == EXPECTED_RECORDS
        )
    },

    {
        "check": "utility_evidence",
        "observed": utility_records,
        "expected": EXPECTED_RECORDS,
        "passed": (
            utility_records == EXPECTED_RECORDS
        )
    },

    {
        "check": "empirical_privacy_evidence",
        "observed": empirical_privacy_records,
        "expected": EXPECTED_RECORDS,
        "passed": (
            empirical_privacy_records == EXPECTED_RECORDS
        )
    },

    {
        "check": "three_way_coverage",
        "observed": complete_three_way_records,
        "expected": EXPECTED_RECORDS,
        "passed": (
            complete_three_way_records == EXPECTED_RECORDS
        )
    },

    {
        "check": "formal_dp_accounting",
        "observed": formal_dp_records,
        "expected": EXPECTED_DP_RECORDS,
        "passed": formal_dp_complete
    },

    {
        "check": "artifact_saving",
        "observed": int(core_outputs_valid),
        "expected": 1,
        "passed": core_outputs_valid
    }

]

validation_df = pd.DataFrame(
    VALIDATION_CHECKS
)

validation_path = (
    VALIDATION_DIR / "final_validation_report.csv"
)

validation_df.to_csv(
    validation_path,
    index=False
)


# ============================================================
# 14. COMPLETION SUMMARY
# ============================================================

summary = {

    "notebook": NOTEBOOK_ID,

    "title": NOTEBOOK_NAME,

    "status": FINAL_STATUS,

    "execution_complete": execution_complete,

    "research_evaluation_complete": (
        research_evaluation_complete
    ),

    "expected_experiments": EXPECTED_RECORDS,

    "aligned_experiments": aligned_records,

    "unique_experimental_conditions": (
        unique_experimental_conditions
    ),

    "fidelity_evidence_records": fidelity_records,

    "utility_evidence_records": utility_records,

    "empirical_privacy_evidence_records": (
        empirical_privacy_records
    ),

    "complete_tradeoff_records": (
        complete_three_way_records
    ),

    "pareto_optimal_records": pareto_records,

    "optimal_operating_points": optimal_records,

    "expected_dp_experiments": EXPECTED_DP_RECORDS,

    "configured_dp_records": configured_dp_records,

    "dp_applied_records": dp_applied_records,

    "formal_dp_evidence_records": formal_dp_records,

    "formal_dp_complete": formal_dp_complete,

    "generated_figures": generated_figure_count,

    "section_17_artifacts_valid": (
        section_17_artifacts_valid
    ),

    "core_outputs_valid": core_outputs_valid,

    "validation_passed": int(
        validation_df["passed"].sum()
    ),

    "validation_failed": int(
        (~validation_df["passed"]).sum()
    ),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

summary_path = (
    METADATA_DIR
    / "notebook_18_completion_summary.json"
)

with open(summary_path, "w") as file:

    json.dump(
        summary,
        file,
        indent=2
    )


# ============================================================
# 15. ARTIFACT INTEGRITY MANIFEST
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


integrity_path = (
    MANIFESTS_DIR / "artifact_integrity_manifest.csv"
)

integrity_records = []

for path in sorted(
    NOTEBOOK_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    if path == integrity_path:
        continue

    integrity_records.append({

        "relative_path": str(
            path.relative_to(NOTEBOOK_ROOT)
        ),

        "size_bytes": path.stat().st_size,

        "sha256": sha256_file(path)

    })


integrity_df = pd.DataFrame(

    integrity_records,

    columns=[
        "relative_path",
        "size_bytes",
        "sha256"
    ]

)

integrity_df.to_csv(
    integrity_path,
    index=False
)


# ============================================================
# 16. SAVE FINAL SUMMARY AGAIN
# ============================================================
# Update integrity manifest information after it is created.

summary["integrity_manifest"] = str(
    integrity_path
)

summary["integrity_manifest_files"] = len(
    integrity_df
)

summary["integrity_manifest_created"] = (
    integrity_path.is_file()
)

with open(summary_path, "w") as file:

    json.dump(
        summary,
        file,
        indent=2
    )


# ============================================================
# 17. MEMORY CLEANUP
# ============================================================

gc.collect()


# ============================================================
# 18. FINAL COMPLETION OUTPUT
# ============================================================

print("=" * 75)

print("NOTEBOOK 18 — COMPLETION SUMMARY")

print("=" * 75)

print("Status                    :", FINAL_STATUS)

print("Execution complete        :", execution_complete)

print(
    "Research evaluation ready :",
    research_evaluation_complete
)

print("-" * 75)

print("Expected experiments      :", EXPECTED_RECORDS)

print("Aligned experiments       :", aligned_records)

print("Fidelity evidence         :", fidelity_records)

print("Utility evidence          :", utility_records)

print(
    "Empirical privacy evidence:",
    empirical_privacy_records
)

print(
    "Complete trade-off records:",
    complete_three_way_records
)

print("Pareto-optimal records    :", pareto_records)

print("Optimal operating points  :", optimal_records)

print("-" * 75)

print("Expected DP experiments  :", EXPECTED_DP_RECORDS)

print("Configured DP records    :", configured_dp_records)

print("DP applied records       :", dp_applied_records)

print("Formal DP evidence       :", formal_dp_records)

print("Formal DP complete       :", formal_dp_complete)

print("Generated figures        :", generated_figure_count)

print("-" * 75)

print(
    "Validation passed        :",
    summary["validation_passed"]
)

print(
    "Validation failed        :",
    summary["validation_failed"]
)

print(
    "Section 17 artifacts valid:",
    section_17_artifacts_valid
)

print(
    "Core outputs valid       :",
    core_outputs_valid
)

print(
    "Integrity manifest files :",
    len(integrity_df)
)

print("-" * 75)

print("Results directory        :", RESULTS_DIR)

print("Tables directory         :", TABLES_DIR)

print("Figures directory        :", FIGURES_DIR)

print("Completion summary       :", summary_path)

print("Validation report        :", validation_path)

print("Integrity manifest       :", integrity_path)

print("=" * 75)


NOTEBOOK 18 — COMPLETION SUMMARY
Status                    : COMPLETED_WITH_MISSING_EVIDENCE
Execution complete        : True
Research evaluation ready : False
---------------------------------------------------------------------------
Expected experiments      : 60
Aligned experiments       : 60
Fidelity evidence         : 0
Utility evidence          : 0
Empirical privacy evidence: 0
Complete trade-off records: 0
Pareto-optimal records    : 0
Optimal operating points  : 0
---------------------------------------------------------------------------
Expected DP experiments  : 30
Configured DP records    : 30
DP applied records       : 30
Formal DP evidence       : 0
Formal DP complete       : False
Generated figures        : 2
---------------------------------------------------------------------------
Validation passed        : 2
Validation failed        : 5
Section 17 artifacts valid: True
Core outputs valid       : True
Integrity manifest files : 53
------------------------------------